# MSc Dissertation: Reliability-Aware Automatic Video Metadata

**Case study:** NVTV public video archive  
**Environment:** Google Colab with a GPU runtime and Google Drive persistence  
**Notebook version:** 4.6 (Colab port)

This notebook generates automatic reference metadata for 30-second archival-video clips and uses independent human annotation to verify the generated descriptive metadata. Calibration-split labels additionally supervise a small LoRA fine-tuning step on two components (CLIP visual tagging and BLIP-VQA indoor/outdoor classification); evaluation-split labels remain strictly held out and are used only to measure real accuracy, never to tune or select the automatic pipeline. The pipeline combines bounded parallel clip creation and metadata inference, dense OCR probing, scene-aware temporal sampling, strict temporally corroborated OCR, heterogeneous pretrained models, an optional cached Gemini multimodal annotator, evidence-status handling, cross-source agreement, reproducibility controls, ablation experiments, perturbation-based robustness analysis, human-labeled accuracy measurement, LoRA fine-tuning and a read-only Flask evidence explorer.

All persistent clips, predictions, evidence files and reports are stored under Google Drive. Model downloads and temporary working copies use the disposable Colab runtime for faster access.

### Expected Drive layout

```text
MyDrive/
└── NVTV_PublicData1/
    ├── VIDEO FILES/
    ├── SYNOPSES/
    └── automatic_ground_truth_v4/   # created/reused by this notebook
```

### Research questions

1. How does scene-aware sampling affect metadata coverage and stability compared with centre-frame and fixed three-frame sampling?
2. How strongly do independent evidence sources agree across metadata fields, after removing self-similarity and empty-output inflation?
3. How robust is automatically generated metadata to realistic visual degradation?
4. How often does a hosted multimodal LLM corroborate the local model families, and how often does adding it change the selected metadata?
5. Does the cross-model agreement score predict human-judged correctness, and does LoRA fine-tuning two pretrained components on calibration-split human labels raise held-out accuracy?

Replace these questions only if your supervisor has approved different wording.


## Methodological rules

- Human annotations cover both splits. Calibration-split labels supervise LoRA fine-tuning only (Section 18C-D); evaluation-split labels are held out from training entirely and are used only to measure accuracy (Section 18B, 18E). Neither split's labels are used to construct, tune or select the *automatic* (fixed-baseline) metadata itself.
- FFprobe fields are directly extracted container observations; descriptive fields are an automated silver standard.
- `agreement_score` means cross-source corroboration. It is not a calibrated probability of factual correctness.
- Self-similarity is excluded from agreement calculations.
- Empty outputs are represented as `not_detected` or `not_applicable`; they are never promoted to high agreement.
- OCR candidates must persist across frames and receive support from both Tesseract and EasyOCR; otherwise the field is empty or `conflict`.
- Gemini is an optional third visual annotator. It never replaces the strict dual-engine OCR decision and never acts as an authority by itself.
- One source cannot establish agreement. Such a field is retained with `agreement_score = null` and `needs_caution = true`.
- Model-family views receive equal total weight so that extra frames do not create fake independent voters.
- CPU OCR is bounded and parallel; shared local model/GPU inference is serialized to prevent CUDA contention and non-thread-safe model use.
- Gemini requests are sequential, cached and provenance-tracked; the API key is never written to an output file.
- Freeze the model set, prompt, response schema, vocabulary, frame policy, thresholds and benchmark-derived weights before final analysis.
- Accuracy, precision and recall are computed only against the human reference file (Section 18), never against the automatically generated file itself.
- LoRA fine-tuning targets are scoped to a single tower or sub-module per model (CLIP's vision tower only; BLIP-VQA's answer decoder only), so the model's other, still-frozen half stays identical to the version documented in Section 7/13, and adapters are trained only on calibration-split frames.

The pipeline is resumable and writes clips, predictions, evidence, reports and audits to Google Drive.


## 1. Mount Drive and install the Colab environment

In Colab, first select **Runtime → Change runtime type → GPU**. Run the cells in order. The setup installs FFmpeg and Tesseract through `apt` and installs the pinned Python dependencies into the current Colab runtime; it deliberately keeps Colab's CUDA-enabled PyTorch build.

Google Drive is mounted at `/content/drive`. Persistent outputs are written to `MyDrive/NVTV_PublicData1/automatic_ground_truth_v4`, so completed checkpoints survive runtime resets.

For optional Gemini calls, add `GEMINI_API_KEY` in Colab's **Secrets** panel and grant notebook access. The key is read into memory only and is never saved in an output artifact.


In [2]:
import os
from pathlib import Path

from google.colab import drive, userdata

DRIVE_MOUNT = Path("/content/drive")
drive.mount(str(DRIVE_MOUNT), force_remount=False)

# Optional: read Gemini from Colab Secrets without placing the key in the notebook.
try:
    gemini_key = (userdata.get("GEMINI_API_KEY") or "").strip()
except Exception:
    gemini_key = ""

if gemini_key:
    os.environ["GEMINI_API_KEY"] = gemini_key

print("Drive mounted:", DRIVE_MOUNT.is_dir())
print("Gemini key available:", bool(os.environ.get("GEMINI_API_KEY", "").strip()))


Mounted at /content/drive
Drive mounted: True
Gemini key available: True


In [3]:
# One editable path controls every persistent input and output location.
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/NVTV_PublicData1")

print("Drive project root:", DRIVE_PROJECT_ROOT)
print("Project root exists:", DRIVE_PROJECT_ROOT.is_dir())


Drive project root: /content/drive/MyDrive/NVTV_PublicData1
Project root exists: True


In [4]:
import importlib
import importlib.metadata as importlib_metadata
import os
import shutil
import subprocess
import sys


def run_setup(command: list[str]) -> None:
    """Run a Colab setup command and expose useful output on failure."""
    print("\nRunning:", " ".join(command))
    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env={
            **os.environ,
            "DEBIAN_FRONTEND": "noninteractive",
            "PIP_DISABLE_PIP_VERSION_CHECK": "1",
        },
    )
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        raise RuntimeError(
            f"Setup command failed with exit code {result.returncode}:\n"
            f"{' '.join(command)}"
        )


# Colab permits apt installation. Install the command-line tools used by
# clip creation, media inspection, Whisper and OCR.
run_setup(["apt-get", "update", "-qq"])
run_setup([
    "apt-get", "install", "-y", "-qq",
    "ffmpeg", "tesseract-ocr",
])

# Keep Colab's preinstalled CUDA-enabled torch. Install only project packages.
PACKAGES = [
    "setuptools-rust",
    "openai-whisper==20250625",
    "transformers==4.50.3",
    "sentence-transformers==3.4.1",
    "keybert==0.9.0",
    "jiwer==4.0.0",
    "pytesseract==0.3.13",
    "easyocr==1.7.2",
    "yake==0.6.0",
    "scenedetect==0.7.1",
    "jsonschema==4.26.0",
    "flask==3.1.3",
    "pydantic==2.12.5",
    "google-genai==2.17.0",
    "seaborn==0.13.2",
]

run_setup([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "--prefer-binary",
    "--upgrade",
    "--upgrade-strategy",
    "only-if-needed",
    "--progress-bar",
    "off",
    *PACKAGES,
])

print("\nSystem executable checks:")
for executable in ("ffmpeg", "ffprobe", "tesseract"):
    executable_path = shutil.which(executable)
    if executable_path is None:
        raise RuntimeError(f"Required executable is missing: {executable}")
    print(f"  {executable}: {executable_path}")

IMPORT_CHECKS = {
    "whisper": "openai-whisper",
    "transformers": "transformers",
    "sentence_transformers": "sentence-transformers",
    "keybert": "keybert",
    "jiwer": "jiwer",
    "pytesseract": "pytesseract",
    "easyocr": "easyocr",
    "yake": "yake",
    "scenedetect": "scenedetect",
    "jsonschema": "jsonschema",
    "flask": "flask",
    "pydantic": "pydantic",
    "google.genai": "google-genai",
    "seaborn": "seaborn",
}

print("\nPython import checks:")
failed_imports = []
for module_name, distribution_name in IMPORT_CHECKS.items():
    try:
        importlib.import_module(module_name)
        version = importlib_metadata.version(distribution_name)
        print(f"  OK: {module_name:<24} {version}")
    except Exception as error:
        failed_imports.append(f"{module_name}: {type(error).__name__}: {error}")
        print(f"  FAILED: {module_name}: {type(error).__name__}: {error}")

if failed_imports:
    raise RuntimeError(
        "One or more dependency imports failed:\n" + "\n".join(failed_imports)
    )

print("\nSETUP COMPLETED SUCCESSFULLY.")



Running: apt-get update -qq
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)

Running: apt-get install -y -qq ffmpeg tesseract-ocr

Running: /usr/bin/python3 -m pip install --no-cache-dir --prefer-binary --upgrade --upgrade-strategy only-if-needed --progress-bar off setuptools-rust openai-whisper==20250625 transformers==4.50.3 sentence-transformers==3.4.1 keybert==0.9.0 jiwer==4.0.0 pytesseract==0.3.13 easyocr==1.7.2 yake==0.6.0 scenedetect==0.7.1 jsonschema==4.26.0 flask==3.1.3 pydantic==2.12.5 google-genai==2.17.0 seaborn==0.13.2
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getti

/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


  OK: scenedetect              0.7.1
  OK: jsonschema               4.26.0
  OK: flask                    3.1.3
  OK: pydantic                 2.12.5
  OK: google.genai             2.17.0
  OK: seaborn                  0.13.2

SETUP COMPLETED SUCCESSFULLY.


## 2. Imports, deterministic settings and runtime check

In [5]:
# Seaborn is installed in the main setup cell above.
import seaborn as sns
print("Seaborn:", sns.__version__)


Seaborn: 0.13.2


In [6]:
from __future__ import annotations

import gc
import io
import hashlib
import importlib.metadata as importlib_metadata
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import tempfile
import threading
import time
import traceback

os.environ.setdefault("OMP_THREAD_LIMIT", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from difflib import SequenceMatcher
from fractions import Fraction
from pathlib import Path
from typing import Any, Iterable, Optional

import cv2
cv2.setNumThreads(1)

import jiwer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytesseract
import seaborn as sns
import torch
from IPython.display import Video, display
from PIL import Image, ImageEnhance, ImageFilter
from tqdm.auto import tqdm

SEED = 40490925
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
MODEL_INFERENCE_LOCK = threading.Lock()

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CPU mode is supported but the full pipeline will be slow.")

if DEVICE != "cuda":
    print("WARNING: Enable a GPU using Runtime > Change runtime type > GPU before the full run.")

for executable in ("ffmpeg", "ffprobe", "tesseract"):
    if shutil.which(executable) is None:
        raise RuntimeError(
            f"Required executable is missing: {executable}. Rerun the setup cell."
        )


Python: 3.13.15
PyTorch: 2.11.0+cu128
Device: cuda
GPU: NVIDIA A100-SXM4-40GB


## 3. Google Drive configuration and optional Gemini secret

The project root is `/content/drive/MyDrive/NVTV_PublicData1`. Change `DRIVE_PROJECT_ROOT` in the earlier mount cell if your Drive layout differs. All persistent outputs are written below `automatic_ground_truth_v4`; existing checkpoints are reused because `force_reprocess` and `force_rebuild_clips` default to `False`.

Use `max_videos`, `max_clips` or `gemini_max_clips` for a small development run. Keep them as `None` for the complete corpus.

To enable Gemini, add `GEMINI_API_KEY` through Colab Secrets. If the secret is unavailable, new Gemini calls are skipped and the local pipeline plus any valid cached Gemini records continue normally.

Gemini sends selected frames to an external API. Confirm that the video licence and your university ethics/data-governance rules permit this before enabling it.


In [7]:
from pathlib import Path

root = DRIVE_PROJECT_ROOT

if not root.is_dir():
    raise FileNotFoundError(
        f"Drive project root not found: {root}. "
        "Create it or change DRIVE_PROJECT_ROOT in the mount cell."
    )

print("Resolved Drive root:", root.resolve())
for item in sorted(root.iterdir()):
    print(repr(item.name), "directory" if item.is_dir() else "file")


Resolved Drive root: /content/drive/.shortcut-targets-by-id/1Lo--JpETSunZ6JhmeL1Fd5q-bfIugNPV/NVTV_PublicData1
'SYNOPSES' directory
'Untitled folder' directory
'VIDEO FILES' directory
'VIDEO FILES1' directory
'automatic_ground_truth_v4' directory
'automatic_ground_truth_v4_fixed_three' directory
'metadata_out' directory
'output' directory


In [8]:
@dataclass
class Config:
    # Google Drive is the persistent store for inputs, clips and outputs.
    project_root: Path = DRIVE_PROJECT_ROOT

    video_subdir: str = "VIDEO FILES"
    synopsis_subdir: str = "SYNOPSES"
    output_subdir: str = "automatic_ground_truth_v4"

    clip_seconds: float = 30.0
    minimum_final_clip_seconds: float = 2.0

    # Conservative defaults for a typical Colab CPU allocation.
    parallel_clip_creation: bool = True
    clip_workers: int = 2
    ffmpeg_threads_per_worker: int = 1

    parallel_metadata_processing: bool = True
    pipeline_clip_workers: int = 2
    ocr_parallel_workers: int = 2
    checkpoint_every_clips: int = 2

    ocr_upscale_min_width: int = 1280
    ocr_upscale_max_width: int = 1920
    tesseract_min_word_confidence: float = 60.0
    easyocr_min_confidence: float = 0.50
    ocr_within_frame_similarity: float = 0.88
    ocr_temporal_similarity: float = 0.82
    ocr_min_frame_occurrences: int = 2
    ocr_cross_engine_similarity: float = 0.80
    ocr_single_frame_min_confidence: float = 0.85
    ocr_probe_interval_seconds: float = 2.5
    ocr_probe_max_frames: int = 12
    ocr_probe_edge_offset_seconds: float = 0.5

    frame_fractions: tuple[float, ...] = (0.20, 0.50, 0.80)
    frame_sampling_policy: str = "scene_aware"
    max_scene_frames: int = 5
    scene_adaptive_threshold: float = 3.0
    scene_min_length: str = "0.6s"

    ablation_max_clips: int = 8
    robustness_max_clips: int = 5

    use_gemini: bool = True
    gemini_model: str = "gemini-3.6-flash"
    gemini_max_clips: Optional[int] = None
    gemini_max_frames: int = 5
    gemini_image_max_side: int = 1600
    gemini_jpeg_quality: int = 90
    gemini_inline_request_limit_mb: float = 18.0
    gemini_max_retries: int = 5
    gemini_retry_base_seconds: float = 2.0
    gemini_request_interval_seconds: float = 1.0
    gemini_fail_fast: bool = False
    force_reprocess_gemini: bool = False

    tag_top_k: int = 5
    calibration_fraction: float = 0.20

    whisper_model: str = "small"
    vqa_model: str = "Salesforce/blip-vqa-base"
    clip_model: str = "openai/clip-vit-base-patch32"
    semantic_model: str = "sentence-transformers/all-MiniLM-L6-v2"

    verifier_whisper_model: str = "turbo"
    verifier_vqa_model: str = "dandelin/vilt-b32-finetuned-vqa"
    verifier_clip_model: str = "openai/clip-vit-large-patch14"
    verifier_people_model: str = "facebook/detr-resnet-50"

    # Preserve completed Drive checkpoints when Colab sessions are restarted.
    force_rebuild_clips: bool = False
    force_reprocess: bool = False
    fail_fast: bool = False

    # Process every available source video and clip.
    max_videos: Optional[int] = None
    max_clips: Optional[int] = None

    @property
    def video_dir(self) -> Path:
        return self.project_root / self.video_subdir

    @property
    def synopsis_dir(self) -> Path:
        return self.project_root / self.synopsis_subdir

    @property
    def artifact_dir(self) -> Path:
        return self.project_root / self.output_subdir

    @property
    def clip_dir(self) -> Path:
        return self.artifact_dir / "clips"

    @property
    def frame_dir(self) -> Path:
        return self.artifact_dir / "keyframes"

    @property
    def manifest_file(self) -> Path:
        return self.artifact_dir / "clip_manifest.csv"


CFG = Config()


# ------------------------------------------------------------
# Resolve bounded Colab CPU parallelism.
# ------------------------------------------------------------

available_cpus = max(1, os.cpu_count() or 1)
configured_ffmpeg_cpus = CFG.clip_workers * CFG.ffmpeg_threads_per_worker
if configured_ffmpeg_cpus > available_cpus:
    raise RuntimeError(
        f"FFmpeg is configured for {configured_ffmpeg_cpus} CPUs, "
        f"but this Colab runtime exposes only {available_cpus}."
    )


# ------------------------------------------------------------
# Validate inputs before creating output directories.
# ------------------------------------------------------------

if not CFG.project_root.is_dir():
    raise FileNotFoundError(
        f"Project root not found: {CFG.project_root}"
    )

if not CFG.video_dir.is_dir():
    raise FileNotFoundError(
        f"Video folder not found: {CFG.video_dir}"
    )

if not CFG.synopsis_dir.is_dir():
    raise FileNotFoundError(
        f"Synopsis folder not found: {CFG.synopsis_dir}"
    )


# ------------------------------------------------------------
# Create persistent output directories.
# ------------------------------------------------------------

for folder in (
    CFG.artifact_dir,
    CFG.clip_dir,
    CFG.frame_dir,
):
    folder.mkdir(parents=True, exist_ok=True)


VISUAL_LABELS = (
    "news studio",
    "interview",
    "press conference",
    "public meeting",
    "panel discussion",
    "person speaking",
    "crowd of people",
    "protest",
    "indoor scene",
    "outdoor scene",
    "city street",
    "building exterior",
    "office",
    "stage",
    "podium",
    "fire engine",
    "emergency services",
    "police",
    "hospital",
    "school",
    "graphic or title card",
    "text on screen",
    "logo",
    "presentation slide",
    "landscape",
    "rural countryside",
    "vehicle",
    "road",
    "sign or banner",
    "audience",
    "reporter",
    "politician",
    "firefighter",
    "uniform",
)

VQA_QUESTIONS = {
    "people_count": "How many people are visible?",
}


# ------------------------------------------------------------
# Node-local temporary cache.
# ------------------------------------------------------------

LOCAL_CACHE = (
    Path(os.environ.get("TMPDIR", tempfile.gettempdir()))
    / "nvtv_dissertation_cache"
)
LOCAL_CLIP_CACHE = LOCAL_CACHE / "clips"
LOCAL_CLIP_CACHE.mkdir(parents=True, exist_ok=True)


print("Project root:", CFG.project_root)
print("Video folder:", CFG.video_dir)
print("Synopsis folder:", CFG.synopsis_dir)
print("Persistent outputs:", CFG.artifact_dir)
print("Colab runtime CPUs:", available_cpus)
print("Persistent outputs are on Drive:", str(CFG.artifact_dir).startswith("/content/drive/"))
print(
    "FFmpeg parallelism:",
    f"{CFG.clip_workers} workers × "
    f"{CFG.ffmpeg_threads_per_worker} threads = "
    f"{configured_ffmpeg_cpus} CPUs",
)


Project root: /content/drive/MyDrive/NVTV_PublicData1
Video folder: /content/drive/MyDrive/NVTV_PublicData1/VIDEO FILES
Synopsis folder: /content/drive/MyDrive/NVTV_PublicData1/SYNOPSES
Persistent outputs: /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4
Colab runtime CPUs: 12
Persistent outputs are on Drive: True
FFmpeg parallelism: 2 workers × 1 threads = 2 CPUs


## 4. Discover source videos

In [9]:
VIDEO_EXTENSIONS = {
    ".mp4", ".mov", ".mkv", ".avi", ".m4v",
    ".webm", ".mpg", ".mpeg",
}

VIDEO_PATHS = sorted(
    path for path in CFG.video_dir.rglob("*")
    if path.is_file()
    and path.suffix.lower() in VIDEO_EXTENSIONS
    and not path.name.startswith("._")
)
if CFG.max_videos is not None:
    VIDEO_PATHS = VIDEO_PATHS[: CFG.max_videos]

if not VIDEO_PATHS:
    raise FileNotFoundError(f"No supported videos found under {CFG.video_dir}")

print(f"Found {len(VIDEO_PATHS)} source video(s).")
for path in VIDEO_PATHS:
    print(" -", path.relative_to(CFG.video_dir))

Found 47 source video(s).
 - 1. Launch of Belfast Children's Festival 2016 280116.mp4
 - 10. Jacobin Launch 230316.mp4
 - 11. Poor Mental Health on Peace Lines 150416.mp4
 - 12. Oliver Pollock Blue Plaque Unveiling 150416.mp4
 - 13. Reading Rooms at Open University 150416.mp4
 - 14. Kabosh - 1916 Community Play 150416.mp4
 - 15. May Day Parade 020516.mp4
 - 16. Political Special 020516.mp4
 - 17. Belfast Telegraph 100516.mp4
 - 18. Fergal McFerran 100516.mp4
 - 19. George McBride 100516.mp4
 - 2. Belfast's City Council - Step Up To Learn Programme 280116.mp4
 - 20. Writers on Writers Festival 100516.mp4
 - 24. Fairtrade Campaign 070616.mp4
 - 25. Blue Plaque Unveiling 010716.mp4
 - 26. Sixteen South 010716.mp4
 - 27. Growing in a Shared City 010716.mp4
 - 28. Feile an Phobail launch 010716.mp4
 - 29. Blue Plaque Unveiling 200716.mp4
 - 3. Fire Brigades Union Northern Ireland - Threats to the Fire and Rescue Service 280116.mp4
 - 30. Social Policy Association 200716.mp4
 - 31. Lilly and

## 5. Bounded parallel, frame-accurate clip creation

Independent 30-second clip jobs are executed with a bounded thread pool. Each worker launches one FFmpeg process, writes to a unique partial file, validates the duration and atomically publishes the result. Completed clips are reused on later runs.

The Colab default is two workers with one FFmpeg encoding thread each; increase it only after checking `os.cpu_count()`. More workers are not automatically better: excessive parallelism causes CPU oversubscription and shared-filesystem contention. The final manifest is sorted deterministically, so task completion order cannot change dataset ordering or the source-level split.


In [10]:
def run_checked(command: list[str]) -> subprocess.CompletedProcess:
    result = subprocess.run(
        [str(part) for part in command],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {' '.join(map(str, command))}\n"
            f"{result.stderr[-3000:]}"
        )
    return result


def parse_fraction(value: Any) -> float:
    try:
        return float(Fraction(str(value)))
    except (ValueError, ZeroDivisionError):
        return 0.0


def probe_media(path: Path) -> dict[str, Any]:
    result = run_checked([
        "ffprobe", "-v", "error",
        "-show_streams", "-show_format",
        "-of", "json", str(path),
    ])
    payload = json.loads(result.stdout)
    streams = payload.get("streams", [])
    video = next((s for s in streams if s.get("codec_type") == "video"), {})
    audio = next((s for s in streams if s.get("codec_type") == "audio"), {})
    fmt = payload.get("format", {})
    duration = float(fmt.get("duration") or video.get("duration") or 0.0)
    return {
        "duration_sec": duration,
        "width": int(video.get("width") or 0),
        "height": int(video.get("height") or 0),
        "fps": parse_fraction(video.get("avg_frame_rate") or video.get("r_frame_rate") or "0/1"),
        "video_codec": video.get("codec_name"),
        "audio_codec": audio.get("codec_name"),
        "has_audio": bool(audio),
        "bitrate": int(fmt.get("bit_rate") or 0),
        "file_size": int(fmt.get("size") or path.stat().st_size),
        "container": fmt.get("format_name"),
    }


def file_sha256(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_identifier(value: str, max_length: int = 72) -> str:
    clean = re.sub(r"[^A-Za-z0-9]+", "_", value).strip("_")
    return (clean or "video")[:max_length]


def source_identifier(path: Path) -> str:
    relative = str(path.relative_to(CFG.video_dir)).replace("\\", "/")
    source_token = f"{relative}|{path.stat().st_size}"
    digest = hashlib.sha1(source_token.encode("utf-8")).hexdigest()[:8]
    return f"{safe_identifier(path.stem)}_{digest}"


def valid_existing_clip(
    path: Path,
    expected_duration: Optional[float] = None,
) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        actual_duration = probe_media(path)["duration_sec"]
        if actual_duration <= 0.25:
            return False
        if expected_duration is not None:
            return abs(actual_duration - expected_duration) <= 0.75
        return True
    except Exception:
        return False


def write_csv_atomic(frame: pd.DataFrame, path: Path) -> None:
    temp_path = path.with_name(path.stem + ".partial.csv")
    frame.to_csv(temp_path, index=False)
    os.replace(temp_path, path)


def assign_source_splits(source_ids: list[str]) -> dict[str, str]:
    unique_ids = sorted(set(source_ids))
    if len(unique_ids) < 2:
        return {source_id: "evaluation" for source_id in unique_ids}
    rng = np.random.default_rng(SEED)
    shuffled = list(rng.permutation(unique_ids))
    n_calibration = max(1, int(round(len(unique_ids) * CFG.calibration_fraction)))
    n_calibration = min(n_calibration, len(unique_ids) - 1)
    calibration = set(shuffled[:n_calibration])
    return {
        source_id: "calibration" if source_id in calibration else "evaluation"
        for source_id in unique_ids
    }


@dataclass(frozen=True)
class ClipJob:
    source_path: Path
    source_id: str
    source_video: str
    source_sha256: str
    split: str
    source_duration_sec: float
    clip_index: int
    start_sec: float
    intended_duration_sec: float
    clip_id: str
    clip_path: Path


def plan_clip_jobs(
    source_rows: list[tuple[Path, str, dict[str, Any], str]],
    split_map: dict[str, str],
) -> list[ClipJob]:
    jobs = []
    duration_tag = f"{int(round(CFG.clip_seconds * 1000)):06d}ms"
    for source_path, source_id, source_info, source_hash in source_rows:
        total = float(source_info["duration_sec"])
        n_segments = int(math.ceil(total / CFG.clip_seconds))
        for index in range(n_segments):
            if CFG.max_clips is not None and len(jobs) >= CFG.max_clips:
                return jobs
            start = index * CFG.clip_seconds
            intended = min(CFG.clip_seconds, total - start)
            if intended < CFG.minimum_final_clip_seconds:
                continue
            clip_id = f"{source_id}__{duration_tag}__clip{index:04d}"
            jobs.append(ClipJob(
                source_path=source_path,
                source_id=source_id,
                source_video=source_path.name,
                source_sha256=source_hash,
                split=split_map[source_id],
                source_duration_sec=total,
                clip_index=index,
                start_sec=float(start),
                intended_duration_sec=float(intended),
                clip_id=clip_id,
                clip_path=CFG.clip_dir / f"{clip_id}.mp4",
            ))
    return jobs


def resolved_clip_workers(job_count: int) -> int:
    if job_count <= 0:
        return 0
    if not CFG.parallel_clip_creation:
        return 1
    requested = int(CFG.clip_workers)
    if requested < 1:
        raise ValueError("CFG.clip_workers must be at least 1.")
    available_cpus = max(1, int(os.cpu_count() or 1))
    return min(requested, available_cpus, job_count)


def create_or_validate_clip(
    job: ClipJob,
) -> dict[str, Any]:
    clip_path = job.clip_path
    must_build = (
        CFG.force_rebuild_clips
        or not valid_existing_clip(
            clip_path,
            job.intended_duration_sec,
        )
    )
    if must_build:
        partial = clip_path.with_name(
            clip_path.stem + ".partial.mp4"
        )
        partial.unlink(missing_ok=True)
        command = [
            "ffmpeg", "-hide_banner", "-loglevel", "error",
            "-nostdin", "-y",
            "-ss", f"{job.start_sec:.3f}",
            "-i", str(job.source_path),
            "-t", f"{job.intended_duration_sec:.3f}",
            "-map", "0:v:0", "-map", "0:a?",
            "-c:v", "libx264",
            "-threads", str(CFG.ffmpeg_threads_per_worker),
            "-preset", "veryfast", "-crf", "20",
            "-c:a", "aac", "-b:a", "128k",
            "-movflags", "+faststart",
            str(partial),
        ]
        try:
            run_checked(command)
            if not valid_existing_clip(
                partial,
                job.intended_duration_sec,
            ):
                raise RuntimeError(
                    f"Invalid clip produced: {partial}"
                )
            os.replace(partial, clip_path)
        finally:
            partial.unlink(missing_ok=True)

    actual = probe_media(clip_path)
    return {
        "clip_id": job.clip_id,
        "source_id": job.source_id,
        "source_video": job.source_video,
        "source_path": str(job.source_path),
        "source_sha256": job.source_sha256,
        "clip_sha256": file_sha256(clip_path),
        "split": job.split,
        "clip_index": job.clip_index,
        "start_sec": round(job.start_sec, 3),
        "end_sec": round(
            min(
                job.start_sec + actual["duration_sec"],
                job.source_duration_sec,
            ),
            3,
        ),
        "intended_duration_sec": round(
            job.intended_duration_sec, 3
        ),
        "actual_duration_sec": round(
            actual["duration_sec"], 3
        ),
        "clip_path": str(clip_path),
        "creation_status": "created" if must_build else "reused",
    }


def execute_clip_jobs(
    jobs: list[ClipJob],
) -> list[dict[str, Any]]:
    workers = resolved_clip_workers(len(jobs))
    if workers == 0:
        return []
    ffmpeg_threads = int(CFG.ffmpeg_threads_per_worker)
    if ffmpeg_threads < 1:
        raise ValueError(
            "CFG.ffmpeg_threads_per_worker must be at least 1."
        )
    print(
        f"Clip jobs: {len(jobs)} | workers: {workers} | "
        f"FFmpeg threads per worker: {ffmpeg_threads}"
    )

    records, failures = [], []
    if workers == 1:
        for job in tqdm(jobs, desc="Creating 30-second clips"):
            try:
                records.append(create_or_validate_clip(job))
            except Exception as exc:
                failures.append((job.clip_id, exc))
                if CFG.fail_fast:
                    raise
    else:
        with ThreadPoolExecutor(
            max_workers=workers,
            thread_name_prefix="nvtv-ffmpeg",
        ) as executor:
            future_to_job = {
                executor.submit(create_or_validate_clip, job): job
                for job in jobs
            }
            progress = tqdm(
                total=len(jobs),
                desc=f"Creating 30-second clips ({workers} workers)",
            )
            try:
                for future in as_completed(future_to_job):
                    job = future_to_job[future]
                    try:
                        records.append(future.result())
                    except Exception as exc:
                        failures.append((job.clip_id, exc))
                        if CFG.fail_fast:
                            for pending in future_to_job:
                                pending.cancel()
                            raise
                    finally:
                        progress.update(1)
            finally:
                progress.close()

    if failures:
        preview = "\n".join(
            f"- {clip_id}: {type(exc).__name__}: {exc}"
            for clip_id, exc in failures[:8]
        )
        raise RuntimeError(
            f"{len(failures)} clip job(s) failed.\n{preview}"
        )
    return records


def build_clip_manifest(video_paths: list[Path]) -> pd.DataFrame:
    source_rows = []
    for source_path in tqdm(
        video_paths,
        desc="Indexing source videos",
    ):
        source_info = probe_media(source_path)
        if source_info["duration_sec"] <= 0:
            raise ValueError(
                f"Could not determine duration: {source_path}"
            )
        source_rows.append((
            source_path,
            source_identifier(source_path),
            source_info,
            file_sha256(source_path),
        ))

    split_map = assign_source_splits(
        [row[1] for row in source_rows]
    )
    jobs = plan_clip_jobs(source_rows, split_map)

# Let workers initially process different source videos.
    jobs.sort(key=lambda job: (job.clip_index, job.source_id))

    records = execute_clip_jobs(jobs)
    if not records:
        raise RuntimeError(
            "No clips were created. Check the source videos, "
            "minimum clip duration and max_clips setting."
        )

    manifest = pd.DataFrame(records).sort_values(
        ["source_id", "clip_index"]
    ).reset_index(drop=True)
    if manifest["clip_id"].duplicated().any():
        raise AssertionError("Duplicate clip identifiers detected.")
    write_csv_atomic(manifest, CFG.manifest_file)
    counts = manifest["creation_status"].value_counts().to_dict()
    print(
        "Clip creation summary:",
        {key: int(value) for key, value in counts.items()},
    )
    return manifest

In [11]:
def load_cached_manifest() -> Optional[pd.DataFrame]:
    if CFG.force_rebuild_clips or not CFG.manifest_file.is_file():
        return None

    cached = pd.read_csv(CFG.manifest_file)

    required_columns = {
        "clip_id",
        "source_id",
        "source_video",
        "clip_path",
        "creation_status",
    }
    if not required_columns.issubset(cached.columns):
        return None

    expected_source_ids = {
        source_identifier(path) for path in VIDEO_PATHS
    }

    if set(cached["source_id"]) != expected_source_ids:
        print(
            "Cached manifest does not cover the selected "
            "source videos."
        )
        return None

    cached["clip_path"] = cached["clip_id"].map(
        lambda clip_id: str(
            CFG.clip_dir / f"{clip_id}.mp4"
        )
    )

    missing_clips = [
        path
        for path in cached["clip_path"]
        if (
            not Path(path).is_file()
            or Path(path).stat().st_size == 0
        )
    ]

    if missing_clips:
        print(
            f"{len(missing_clips)} cached clips are missing."
        )
        return None

    print(
        f"Loaded cached manifest with {len(cached)} clips."
    )
    return cached


MANIFEST = load_cached_manifest()

if MANIFEST is None:
    MANIFEST = build_clip_manifest(VIDEO_PATHS)

display(
    MANIFEST.groupby(
        ["split", "source_id", "source_video"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "clips"})
)

print(f"Manifest: {CFG.manifest_file}")
print(f"Total clips: {len(MANIFEST)}")
print(
    "Creation status:",
    MANIFEST["creation_status"].value_counts().to_dict(),
)

Loaded cached manifest with 626 clips.


,split,source_id,source_video,clips
0,calibration,15_May_Day_Parade_020516_1f57ff51,15. May Day Parade 020516.mp4,18
1,calibration,19_George_McBride_100516_3392aae5,19. George McBride 100516.mp4,12
2,calibration,1_Launch_of_Belfast_Children_s_Festival_2016_2...,1. Launch of Belfast Children's Festival 2016 ...,16
3,calibration,26_Sixteen_South_010716_99634b07,26. Sixteen South 010716.mp4,10
4,calibration,38_ArtisAnn_Art_Gallery_090816_1b976311,38. ArtisAnn Art Gallery 090816.mp4,14
5,calibration,44_Beat_The_Street_050916_5edd2a4b,44. Beat The Street 050916.mp4,12
6,calibration,45_Relate_NI_050916_75493691,45. Relate NI 050916.mp4,11
7,calibration,4_Ballymena_Protest_040216_fae3fcab,4. Ballymena Protest 040216.mp4,17
8,calibration,9_KESS_Enabling_Access_to_Justice_230316_54a23dc7,9. KESS - Enabling Access to Justice 230316.mp4,13
9,evaluation,10_Jacobin_Launch_230316_13c6f149,10. Jacobin Launch 230316.mp4,16


Manifest: /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4/clip_manifest.csv
Total clips: 626
Creation status: {'reused': 626}


## 7. Load the pretrained models

Model choices are deliberately small enough for a standard Colab GPU runtime. A T4 is sufficient for the baseline model set, although model downloads and the first run can take time:

- Whisper-small: speech recognition and language identification
- Tesseract: on-screen text
- BLIP VQA: visible people
- CLIP ViT-B/32: controlled-vocabulary visual tag ranking
- KeyBERT with MiniLM: transcript keywords

CLIP scores are relative ranking scores, not calibrated probabilities.


In [12]:
import os
from pathlib import Path
from typing import Any, Optional

import torch
import whisper
from huggingface_hub import snapshot_download
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer
from transformers import (
    BlipForQuestionAnswering,
    BlipProcessor,
    CLIPModel,
    CLIPProcessor,
)


# ---------------------------------------------------------
# Store model downloads in the fast, disposable Colab runtime.
# Research outputs remain persistent under CFG.artifact_dir on Drive.
# ---------------------------------------------------------

SCRATCH_ROOT = Path("/content/nvtv_runtime_cache")

HF_HOME_DIR = (
    SCRATCH_ROOT / "huggingface-cache"
)

HF_HUB_CACHE_DIR = (
    HF_HOME_DIR / "hub"
)

HF_XET_CACHE_DIR = (
    HF_HOME_DIR / "xet"
)

WHISPER_CACHE_DIR = (
    SCRATCH_ROOT / "whisper-cache"
)

SEMANTIC_MODEL_DIR = (
    SCRATCH_ROOT
    / "models"
    / "all-MiniLM-L6-v2-complete"
)

for directory in (
    HF_HOME_DIR,
    HF_HUB_CACHE_DIR,
    HF_XET_CACHE_DIR,
    WHISPER_CACHE_DIR,
    SEMANTIC_MODEL_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

os.environ["HF_HOME"] = str(
    HF_HOME_DIR
)

os.environ["HF_HUB_CACHE"] = str(
    HF_HUB_CACHE_DIR
)

os.environ["HUGGINGFACE_HUB_CACHE"] = str(
    HF_HUB_CACHE_DIR
)

os.environ["HF_XET_CACHE"] = str(
    HF_XET_CACHE_DIR
)

os.environ["XDG_CACHE_HOME"] = str(
    SCRATCH_ROOT / "cache"
)


def move_batch(
    batch: Any,
) -> dict[str, torch.Tensor]:
    moved = {}

    for key, value in batch.items():
        if torch.is_tensor(value):
            if value.is_floating_point():
                moved[key] = value.to(
                    device=DEVICE,
                    dtype=MODEL_DTYPE,
                )
            else:
                moved[key] = value.to(
                    device=DEVICE
                )
        else:
            moved[key] = value

    return moved


def tensor_output(
    value: Any,
) -> torch.Tensor:
    if torch.is_tensor(value):
        return value

    if hasattr(value, "pooler_output"):
        return value.pooler_output

    raise TypeError(
        f"Unexpected model output type: {type(value)}"
    )


def resolved_model_revision(
    model: Any,
) -> Optional[str]:
    candidates = [
        model,
        getattr(model, "model", None),
        getattr(model, "auto_model", None),
    ]

    try:
        candidates.append(
            model[0].auto_model
        )
    except Exception:
        pass

    for candidate in candidates:
        config = getattr(
            candidate,
            "config",
            None,
        )

        revision = getattr(
            config,
            "_commit_hash",
            None,
        )

        if revision:
            return str(revision)

    return None


def load_models() -> dict[str, Any]:
    print(
        "Loading Whisper:",
        CFG.whisper_model,
    )

    asr = whisper.load_model(
        CFG.whisper_model,
        device=DEVICE,
        download_root=str(
            WHISPER_CACHE_DIR
        ),
    )

    print(
        "Loading VQA model:",
        CFG.vqa_model,
    )

    vqa_processor = (
        BlipProcessor.from_pretrained(
            CFG.vqa_model,
            cache_dir=str(
                HF_HUB_CACHE_DIR
            ),
            use_fast=False,
        )
    )

    vqa_model = (
        BlipForQuestionAnswering
        .from_pretrained(
            CFG.vqa_model,
            cache_dir=str(
                HF_HUB_CACHE_DIR
            ),
            torch_dtype=MODEL_DTYPE,
        )
        .to(DEVICE)
        .eval()
    )

    print(
        "Loading CLIP:",
        CFG.clip_model,
    )

    clip_processor = (
        CLIPProcessor.from_pretrained(
            CFG.clip_model,
            cache_dir=str(
                HF_HUB_CACHE_DIR
            ),
            use_fast=False,
        )
    )

    clip_model = (
        CLIPModel.from_pretrained(
            CFG.clip_model,
            cache_dir=str(
                HF_HUB_CACHE_DIR
            ),
            torch_dtype=MODEL_DTYPE,
        )
        .to(DEVICE)
        .eval()
    )

    print(
        "Downloading semantic encoder:",
        CFG.semantic_model,
    )

    semantic_model_path = snapshot_download(
        repo_id=CFG.semantic_model,
        local_dir=str(
            SEMANTIC_MODEL_DIR
        ),
        token=False,
    )

    print(
        "Loading semantic encoder from:",
        semantic_model_path,
    )

    semantic_encoder = SentenceTransformer(
        semantic_model_path,
        device=DEVICE,
        local_files_only=True,
    )

    keyword_model = KeyBERT(
        model=semantic_encoder
    )

    label_inputs = clip_processor(
        text=[
            f"a photograph of {label}"
            for label in VISUAL_LABELS
        ],
        return_tensors="pt",
        padding=True,
    )

    with torch.inference_mode():
        text_features = tensor_output(
            clip_model.get_text_features(
                **move_batch(label_inputs)
            )
        )

        text_features = (
            torch.nn.functional.normalize(
                text_features,
                dim=-1,
            )
        )

    return {
        "asr": asr,
        "vqa_processor": vqa_processor,
        "vqa_model": vqa_model,
        "clip_processor": clip_processor,
        "clip_model": clip_model,
        "clip_text_features": text_features,
        "semantic_encoder": semantic_encoder,
        "keyword_model": keyword_model,
    }


MODELS = load_models()

print("All models loaded.")


Loading Whisper: small


100%|████████████████████████████████████████| 461M/461M [00:01<00:00, 393MiB/s]


Loading VQA model: Salesforce/blip-vqa-base


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

train_script.py: 0.00B [00:00, ?B/s]

tf_model.h5:   0%|          | 0.00/91.0M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Loading semantic encoder from: /content/nvtv_runtime_cache/models/all-MiniLM-L6-v2-complete
All models loaded.


## 8. Inference and aggregation helpers

In [13]:
def copy_clip_to_local(persistent_path: Path) -> Path:
    local_path = LOCAL_CLIP_CACHE / persistent_path.name
    if (
        local_path.exists()
        and local_path.stat().st_size == persistent_path.stat().st_size
    ):
        return local_path
    partial = local_path.with_name(local_path.stem + ".partial.mp4")
    partial.unlink(missing_ok=True)
    shutil.copy2(persistent_path, partial)
    os.replace(partial, local_path)
    return local_path


def extract_keyframes(
    clip_path: Path,
    clip_id: str,
    fractions: tuple[float, ...],
) -> list[Image.Image]:
    info = probe_media(clip_path)
    duration = info["duration_sec"]
    if duration <= 0:
        raise ValueError(f"Invalid duration for {clip_path}")

    cap = cv2.VideoCapture(str(clip_path))
    if not cap.isOpened():
        raise RuntimeError(f"OpenCV could not open {clip_path}")

    images = []
    try:
        for frame_index, fraction in enumerate(fractions):
            fraction_tag = int(round(fraction * 1000))
            frame_file = (
                CFG.frame_dir
                / f"{clip_id}__f{frame_index}_{fraction_tag:04d}.jpg"
            )
            if frame_file.exists() and frame_file.stat().st_size > 0:
                images.append(Image.open(frame_file).convert("RGB"))
                continue

            at_sec = min(max(duration * fraction, 0.0), max(duration - 0.05, 0.0))
            cap.set(cv2.CAP_PROP_POS_MSEC, at_sec * 1000.0)
            ok, bgr = cap.read()
            if not ok:
                raise RuntimeError(
                    f"Could not decode keyframe {frame_index} from {clip_path}"
                )
            image = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            partial = frame_file.with_name(frame_file.stem + ".partial.jpg")
            image.save(partial, quality=92)
            os.replace(partial, frame_file)
            images.append(image)
    finally:
        cap.release()

    if len(images) != len(fractions):
        raise AssertionError("Incorrect number of keyframes extracted.")
    return images


def normalize_space(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def normalize_key(value: Any) -> str:
    return re.sub(r"[^a-z0-9]+", " ", normalize_space(value).lower()).strip()


def unique_strings(values: Iterable[str]) -> list[str]:
    output, seen = [], set()
    for value in values:
        clean = normalize_space(value)
        key = normalize_key(clean)
        if clean and key and key not in seen:
            seen.add(key)
            output.append(clean)
    return output


def plausible_ocr_text(value: Any) -> bool:
    text = normalize_space(value)
    if not text or len(text) > 160:
        return False
    non_space = [character for character in text if not character.isspace()]
    alphanumeric = sum(character.isalnum() for character in non_space)
    if not non_space or alphanumeric / len(non_space) < 0.50:
        return False
    key = normalize_key(text)
    return len(key) >= 2 or key.isdigit()



def prepare_ocr_variants(image: Image.Image) -> dict[str, Image.Image]:
    rgb = np.asarray(image.convert("RGB"))
    height, width = rgb.shape[:2]
    if width <= 0 or height <= 0:
        raise ValueError("OCR received an empty image.")

    target_width = min(
        max(width, CFG.ocr_upscale_min_width),
        CFG.ocr_upscale_max_width,
    )
    scale = target_width / width
    interpolation = cv2.INTER_CUBIC if scale > 1.0 else cv2.INTER_AREA
    resized = cv2.resize(
        rgb,
        (target_width, max(1, int(round(height * scale)))),
        interpolation=interpolation,
    )

    # Apply CLAHE only to luminance so coloured captions and graphics are
    # not destroyed by an early greyscale conversion.
    lab = cv2.cvtColor(resized, cv2.COLOR_RGB2LAB)
    luminance, channel_a, channel_b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced_lab = cv2.merge((clahe.apply(luminance), channel_a, channel_b))
    enhanced_rgb = cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2RGB)
    blurred_rgb = cv2.GaussianBlur(enhanced_rgb, (0, 0), sigmaX=1.0)
    sharpened_rgb = cv2.addWeighted(
        enhanced_rgb, 1.7, blurred_rgb, -0.7, 0
    )
    grey = cv2.cvtColor(sharpened_rgb, cv2.COLOR_RGB2GRAY)
    _, otsu = cv2.threshold(
        grey,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU,
    )
    adaptive = cv2.adaptiveThreshold(
        grey,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31,
        11,
    )
    return {
        "rgb_enhanced": Image.fromarray(sharpened_rgb),
        "otsu": Image.fromarray(otsu),
        "adaptive": Image.fromarray(adaptive),
    }


def merge_similar_ocr_lines(
    lines: list[str],
    confidences: list[float],
    similarity_threshold: Optional[float] = None,
) -> tuple[list[str], list[dict[str, Any]]]:
    if len(lines) != len(confidences):
        raise ValueError("OCR lines and confidence values must align.")
    threshold = (
        CFG.ocr_within_frame_similarity
        if similarity_threshold is None
        else float(similarity_threshold)
    )
    clusters = []
    for order, (line, confidence) in enumerate(zip(lines, confidences)):
        clean = normalize_space(line)
        if not plausible_ocr_text(clean):
            continue
        key = normalize_key(clean)
        best_index, best_similarity = None, -1.0
        for index, cluster in enumerate(clusters):
            similarity = SequenceMatcher(
                None, key, cluster["normalized"]
            ).ratio()
            if similarity > best_similarity:
                best_index, best_similarity = index, similarity
        observation = {
            "text": clean,
            "confidence": float(confidence),
            "order": order,
        }
        if best_index is not None and best_similarity >= threshold:
            cluster = clusters[best_index]
            cluster["observations"].append(observation)
            if float(confidence) > cluster["best_confidence"]:
                cluster["text"] = clean
                cluster["normalized"] = key
                cluster["best_confidence"] = float(confidence)
        else:
            clusters.append({
                "text": clean,
                "normalized": key,
                "best_confidence": float(confidence),
                "first_order": order,
                "observations": [observation],
            })
    clusters.sort(key=lambda item: item["first_order"])
    evidence = [{
        "text": cluster["text"],
        "mean_confidence": round(float(np.mean([
            item["confidence"] for item in cluster["observations"]
        ])), 4),
        "variant_observations": len(cluster["observations"]),
    } for cluster in clusters]
    return [item["text"] for item in evidence], evidence


def tesseract_variant_lines(
    image: Image.Image,
    page_segmentation_mode: int,
) -> tuple[list[str], list[float]]:
    data = pytesseract.image_to_data(
        image,
        config=f"--oem 3 --psm {page_segmentation_mode}",
        output_type=pytesseract.Output.DICT,
    )
    grouped: dict[tuple[int, int, int, int], list[tuple[int, str, float]]] = (
        defaultdict(list)
    )
    count = len(data.get("text", []))
    for index in range(count):
        token = normalize_space(data["text"][index])
        try:
            confidence = float(data["conf"][index])
        except (TypeError, ValueError):
            continue
        if (
            confidence < CFG.tesseract_min_word_confidence
            or not plausible_ocr_text(token)
        ):
            continue
        key = (
            int(data.get("page_num", [0] * count)[index]),
            int(data.get("block_num", [0] * count)[index]),
            int(data.get("par_num", [0] * count)[index]),
            int(data.get("line_num", [0] * count)[index]),
        )
        word_order = int(data.get("word_num", [index] * count)[index])
        grouped[key].append((word_order, token, confidence / 100.0))

    lines, confidences = [], []
    for values in grouped.values():
        values.sort(key=lambda item: item[0])
        line = normalize_space(" ".join(item[1] for item in values))
        if plausible_ocr_text(line):
            lines.append(line)
            confidences.append(float(np.mean([item[2] for item in values])))
    return lines, confidences


def ocr_image(image: Image.Image) -> tuple[list[str], list[float]]:
    variants = prepare_ocr_variants(image)
    lines, confidences = [], []
    for variant_name, psm in (
        ("rgb_enhanced", 3),
        ("otsu", 6),
        ("adaptive", 11),
    ):
        variant_lines, variant_confidences = tesseract_variant_lines(
            variants[variant_name], psm
        )
        lines.extend(variant_lines)
        confidences.extend(variant_confidences)
    merged_lines, evidence = merge_similar_ocr_lines(lines, confidences)
    return (
        merged_lines,
        [float(item["mean_confidence"]) for item in evidence],
    )


def aggregate_temporal_ocr(
    frame_outputs: list[dict[str, Any]],
) -> tuple[list[str], list[dict[str, Any]]]:
    if not frame_outputs:
        return [], []
    clusters = []
    for frame_index, output in enumerate(frame_outputs):
        lines = output.get("on_screen_text", [])
        confidences = output.get(
            "ocr_confidences", [0.0] * len(lines)
        )
        if len(confidences) != len(lines):
            confidences = [0.0] * len(lines)
        for line, confidence in zip(lines, confidences):
            clean = normalize_space(line)
            if not plausible_ocr_text(clean):
                continue
            key = normalize_key(clean)
            best_index, best_similarity = None, -1.0
            for index, cluster in enumerate(clusters):
                similarity = SequenceMatcher(
                    None, key, cluster["normalized"]
                ).ratio()
                if similarity > best_similarity:
                    best_index, best_similarity = index, similarity
            observation = {
                "frame_index": frame_index,
                "text": clean,
                "confidence": float(confidence),
            }
            if (
                best_index is not None
                and best_similarity >= CFG.ocr_temporal_similarity
            ):
                cluster = clusters[best_index]
                existing = next((
                    item for item in cluster["observations"]
                    if item["frame_index"] == frame_index
                ), None)
                if existing is None:
                    cluster["observations"].append(observation)
                elif observation["confidence"] > existing["confidence"]:
                    existing.update(observation)
                if observation["confidence"] > cluster["best_confidence"]:
                    cluster["text"] = clean
                    cluster["normalized"] = key
                    cluster["best_confidence"] = observation["confidence"]
            else:
                clusters.append({
                    "text": clean,
                    "normalized": key,
                    "best_confidence": float(confidence),
                    "first_frame": frame_index,
                    "observations": [observation],
                })

    required_frames = (
        1 if len(frame_outputs) == 1
        else min(CFG.ocr_min_frame_occurrences, len(frame_outputs))
    )
    evidence = []
    for cluster in sorted(clusters, key=lambda item: item["first_frame"]):
        frames = sorted({
            item["frame_index"] for item in cluster["observations"]
        })
        confidence_values = [
            item["confidence"] for item in cluster["observations"]
        ]
        mean_confidence = float(np.mean(confidence_values))
        maximum_confidence = float(np.max(confidence_values))
        strong_single_frame_candidate = (
            len(frames) == 1
            and maximum_confidence >= CFG.ocr_single_frame_min_confidence
            and len(cluster["text"].strip()) >= 4
        )
        accepted = len(frames) >= required_frames or strong_single_frame_candidate
        evidence.append({
            "text": cluster["text"],
            "frames_detected": frames,
            "frame_occurrences": len(frames),
            "required_frame_occurrences": required_frames,
            "mean_confidence": round(mean_confidence, 4),
            "maximum_confidence": round(maximum_confidence, 4),
            "strong_single_frame_candidate": strong_single_frame_candidate,
            "accepted": accepted,
            "rejection_reason": (
                None if accepted else "insufficient_temporal_support"
            ),
        })
    return (
        [item["text"] for item in evidence if item["accepted"]],
        evidence,
    )


@torch.inference_mode()
def answer_visual_question(image: Image.Image, question: str) -> str:
    inputs = MODELS["vqa_processor"](
        images=image,
        text=question,
        return_tensors="pt",
    )
    generated = MODELS["vqa_model"].generate(
        **move_batch(inputs),
        max_new_tokens=12,
    )
    return normalize_space(
        MODELS["vqa_processor"].decode(
            generated[0],
            skip_special_tokens=True,
        )
    )


@torch.inference_mode()
def clip_tag_scores(image: Image.Image) -> dict[str, float]:
    inputs = MODELS["clip_processor"](
        images=image,
        return_tensors="pt",
    )
    image_features = tensor_output(
        MODELS["clip_model"].get_image_features(**move_batch(inputs))
    )
    image_features = torch.nn.functional.normalize(image_features, dim=-1)
    similarities = (
        image_features @ MODELS["clip_text_features"].T
    ).squeeze(0)
    relative_scores = torch.softmax(similarities * 100.0, dim=-1)
    return {
        label: float(score)
        for label, score in zip(VISUAL_LABELS, relative_scores.cpu())
    }


def top_tags(score_map: dict[str, float], top_k: int) -> list[str]:
    return [
        label for label, _ in sorted(
            score_map.items(),
            key=lambda item: item[1],
            reverse=True,
        )[:top_k]
    ]


def extract_keywords(text: str, top_n: int = 8) -> list[str]:
    if not normalize_space(text):
        return []
    candidates = MODELS["keyword_model"].extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words="english",
        use_mmr=True,
        diversity=0.7,
        top_n=top_n * 2,
    )
    selected, used_words = [], set()
    for phrase, _ in candidates:
        words = set(normalize_key(phrase).split())
        if not words:
            continue
        if used_words and len(words & used_words) >= max(1, math.ceil(len(words) / 2)):
            continue
        selected.append(normalize_space(phrase))
        used_words.update(words)
        if len(selected) == top_n:
            break
    return selected


def transcribe_clip(path: Path, has_audio: bool) -> dict[str, Any]:
    if not has_audio:
        return {"text": "", "language": "", "segments": [], "no_audio": True}
    result = MODELS["asr"].transcribe(
        str(path),
        task="transcribe",
        fp16=(DEVICE == "cuda"),
        temperature=0.0,
        condition_on_previous_text=False,
        verbose=False,
    )
    segments = [
        {
            "start": round(float(segment["start"]), 3),
            "end": round(float(segment["end"]), 3),
            "text": normalize_space(segment["text"]),
        }
        for segment in result.get("segments", [])
    ]
    return {
        "text": normalize_space(result.get("text", "")),
        "language": normalize_space(result.get("language", "")),
        "segments": segments,
        "no_audio": False,
    }


NUMBER_WORDS = {
    "zero": 0, "none": 0, "nobody": 0, "one": 1, "single": 1,
    "two": 2, "couple": 2, "three": 3, "few": 3, "four": 4,
    "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9,
    "ten": 10,
}


def parse_people_count(value: Any) -> Optional[int]:
    text = normalize_key(value)
    match = re.search(r"\b\d+\b", text)
    if match:
        return int(match.group())
    for token in text.split():
        if token in NUMBER_WORDS:
            return NUMBER_WORDS[token]
    return None


def mode_or_middle(values: list[str]) -> str:
    clean = [normalize_space(value) for value in values if normalize_space(value)]
    if not clean:
        return ""
    keys = [normalize_key(value) for value in clean]
    counts = Counter(keys)
    best_key, best_count = counts.most_common(1)[0]
    if best_count > 1:
        return clean[keys.index(best_key)]
    return clean[len(clean) // 2]


def aggregate_people(values: list[str]) -> tuple[str, Optional[int]]:
    parsed = [parse_people_count(value) for value in values]
    available = [value for value in parsed if value is not None]
    if not available:
        raw = mode_or_middle(values)
        return raw, parse_people_count(raw)
    median = int(round(float(np.median(available))))
    return str(median), median


def aggregate_frame_outputs(frame_outputs: list[dict[str, Any]]) -> dict[str, Any]:
    if not frame_outputs:
        raise ValueError("At least one frame output is required.")

    mean_scores = {
        label: float(np.mean([
            output["visual_tag_scores"][label] for output in frame_outputs
        ]))
        for label in VISUAL_LABELS
    }
    people_raw, people_numeric = aggregate_people([
        output["facts"]["people_count"] for output in frame_outputs
    ])
    ocr_lines, ocr_temporal_evidence = aggregate_temporal_ocr(
        frame_outputs
    )

    return {
        "on_screen_text": ocr_lines,
        "ocr_temporal_evidence": ocr_temporal_evidence,
        "visual_tags": top_tags(mean_scores, CFG.tag_top_k),
        "visual_tag_scores": mean_scores,
        "people_count": people_raw,
        "people_count_numeric": people_numeric,
    }


def infer_frame_visual_only(image: Image.Image) -> dict[str, Any]:
    return {
        "visual_tag_scores": clip_tag_scores(image),
        "facts": {
            field: answer_visual_question(image, question)
            for field, question in VQA_QUESTIONS.items()
        },
    }


def infer_frame(image: Image.Image) -> dict[str, Any]:
    text_lines, text_confidences = ocr_image(image)
    return {
        "on_screen_text": text_lines,
        "ocr_confidences": text_confidences,
        **infer_frame_visual_only(image),
    }

### Scene-aware temporal sampling

The production policy detects shot boundaries with an adaptive rolling-content detector, samples scene midpoints and always includes the temporal centre. If detection fails or finds too few scenes, fixed 20/50/80% frames are added. The number of frames is capped for GPU memory and runtime feasibility. Fixed three-frame sampling remains available for the ablation experiment.

In [14]:
from scenedetect import AdaptiveDetector, detect

FRAME_SELECTION_CACHE: dict[str, dict[str, Any]] = {}


def deduplicate_times(
    values: Iterable[float],
    duration: float,
    tolerance: float = 0.08,
) -> list[float]:
    upper = max(duration - 0.05, 0.0)
    output = []
    for value in sorted(min(max(float(v), 0.0), upper) for v in values):
        if not output or abs(value - output[-1]) > tolerance:
            output.append(value)
    return output


def limit_frame_times(
    values: list[float],
    maximum: int,
    duration: float,
) -> list[float]:
    values = deduplicate_times(values, duration)
    if len(values) <= maximum:
        return values
    indices = np.linspace(0, len(values) - 1, maximum)
    selected = [values[int(round(index))] for index in indices]
    centre = duration * 0.5
    closest = min(range(len(selected)), key=lambda i: abs(selected[i] - centre))
    selected[closest] = centre
    return deduplicate_times(selected, duration)


def choose_frame_times(
    clip_path: Path,
    duration: float,
    policy: str,
) -> tuple[list[float], dict[str, Any]]:
    fixed = [duration * fraction for fraction in CFG.frame_fractions]
    if policy == "fixed_three":
        times = deduplicate_times(fixed, duration)
        return times, {
            "policy": "fixed_three",
            "fallback_used": False,
            "detected_scene_count": None,
        }
    if policy != "scene_aware":
        raise ValueError(f"Unknown frame sampling policy: {policy}")

    fallback_used = False
    error_message = None
    scene_times = []
    detected_count = 0
    try:
        scenes = detect(
            str(clip_path),
            AdaptiveDetector(
                adaptive_threshold=CFG.scene_adaptive_threshold,
                min_scene_len=CFG.scene_min_length,
            ),
            show_progress=False,
        )
        detected_count = len(scenes)
        scene_times = [
            (start.get_seconds() + end.get_seconds()) / 2.0
            for start, end in scenes
            if end.get_seconds() > start.get_seconds()
        ]
    except Exception as exc:
        fallback_used = True
        error_message = f"{type(exc).__name__}: {exc}"

    candidates = [*scene_times, duration * 0.5]
    if len(deduplicate_times(candidates, duration)) < 3:
        candidates.extend(fixed)
        fallback_used = True
    times = limit_frame_times(
        candidates,
        CFG.max_scene_frames,
        duration,
    )
    return times, {
        "policy": "scene_aware",
        "fallback_used": fallback_used,
        "detected_scene_count": detected_count,
        "detector": "PySceneDetect AdaptiveDetector",
        "adaptive_threshold": CFG.scene_adaptive_threshold,
        "min_scene_length": CFG.scene_min_length,
        "error": error_message,
    }


def extract_keyframes_at_times(
    clip_path: Path,
    clip_id: str,
    times_sec: list[float],
    cache_tag: str,
) -> list[Image.Image]:
    cap = cv2.VideoCapture(str(clip_path))
    if not cap.isOpened():
        raise RuntimeError(f"OpenCV could not open {clip_path}")
    images = []
    try:
        for index, at_sec in enumerate(times_sec):
            millis = int(round(at_sec * 1000))
            frame_file = (
                CFG.frame_dir
                / f"{clip_id}__{cache_tag}_{index:02d}_{millis:07d}ms.jpg"
            )
            if frame_file.exists() and frame_file.stat().st_size > 0:
                with Image.open(frame_file) as cached:
                    images.append(cached.convert("RGB"))
                continue
            cap.set(cv2.CAP_PROP_POS_MSEC, at_sec * 1000.0)
            ok, bgr = cap.read()
            if not ok:
                raise RuntimeError(
                    f"Could not decode frame {index} at {at_sec:.3f}s"
                )
            image = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            partial = frame_file.with_name(
                frame_file.stem + ".partial.jpg"
            )
            image.save(partial, quality=92)
            os.replace(partial, frame_file)
            images.append(image)
    finally:
        cap.release()
    if len(images) != len(times_sec):
        raise AssertionError("Incorrect number of keyframes extracted.")
    return images


def extract_keyframes(
    clip_path: Path,
    clip_id: str,
    fractions: tuple[float, ...],
) -> list[Image.Image]:
    info = probe_media(clip_path)
    duration = float(info["duration_sec"])
    if duration <= 0:
        raise ValueError(f"Invalid duration for {clip_path}")
    times, details = choose_frame_times(
        clip_path,
        duration,
        CFG.frame_sampling_policy,
    )
    details = {
        **details,
        "duration_sec": duration,
        "frame_times_sec": [round(value, 4) for value in times],
        "frame_count": len(times),
    }
    FRAME_SELECTION_CACHE[clip_id] = details
    return extract_keyframes_at_times(
        clip_path,
        clip_id,
        times,
        cache_tag=CFG.frame_sampling_policy,
    )


def choose_ocr_probe_times(
    duration: float,
    scene_frame_times: Iterable[float],
) -> list[float]:
    duration = float(duration)
    if duration <= 0:
        raise ValueError("OCR sampling requires a positive duration.")
    interval = float(CFG.ocr_probe_interval_seconds)
    maximum = int(CFG.ocr_probe_max_frames)
    if interval <= 0 or maximum < 1:
        raise ValueError("OCR probe interval and maximum must be positive.")

    edge = min(
        max(float(CFG.ocr_probe_edge_offset_seconds), 0.0),
        max(duration * 0.5 - 0.05, 0.0),
    )
    last = max(duration - edge, edge)
    uniform = np.arange(
        edge,
        last + interval * 0.25,
        interval,
        dtype=float,
    ).tolist()
    candidates = [
        *uniform,
        *[float(value) for value in scene_frame_times],
        duration * 0.5,
    ]
    return limit_frame_times(candidates, maximum, duration)


def extract_ocr_probe_frames(
    clip_path: Path,
    clip_id: str,
    scene_frame_times: Iterable[float],
) -> tuple[list[Image.Image], list[float]]:
    duration = float(probe_media(clip_path)["duration_sec"])
    times = choose_ocr_probe_times(duration, scene_frame_times)
    images = extract_keyframes_at_times(
        clip_path,
        clip_id,
        times,
        cache_tag="ocr_dense",
    )
    return images, times


## 9. Run the metadata pipeline

For each clip, the notebook performs ASR once and visual inference on three frames. The centre frame is retained as the baseline; all three frames are aggregated as the proposed method.

Predictions are checkpointed after every clip. A failure is recorded with its clip identifier instead of being hidden.

In [15]:
PIPELINE_CONFIG = {
    "schema_version": "4.0",
    "seed": SEED,
    "clip_seconds": CFG.clip_seconds,
    "frame_fractions": list(CFG.frame_fractions),
    "frame_sampling": {
        "policy": CFG.frame_sampling_policy,
        "max_scene_frames": CFG.max_scene_frames,
        "scene_adaptive_threshold": CFG.scene_adaptive_threshold,
        "scene_min_length": CFG.scene_min_length,
    },
    "parallel_execution": {
        "enabled": CFG.parallel_metadata_processing,
        "clip_workers": CFG.pipeline_clip_workers,
        "ocr_workers": CFG.ocr_parallel_workers,
        "checkpoint_every_clips": CFG.checkpoint_every_clips,
        "shared_model_inference_serialized": True,
    },
    "tag_top_k": CFG.tag_top_k,
    "visual_labels": list(VISUAL_LABELS),
    "vqa_questions": VQA_QUESTIONS,
    "ocr_configuration": {
        "preprocessing": [
            "RGB upscaling",
            "colour-luminance CLAHE",
            "unsharp mask",
            "Otsu threshold",
            "adaptive threshold",
        ],
        "page_segmentation_modes": [3, 6, 11],
        "minimum_word_confidence": CFG.tesseract_min_word_confidence,
        "minimum_frame_occurrences": CFG.ocr_min_frame_occurrences,
        "single_frame_candidate_confidence": (
            CFG.ocr_single_frame_min_confidence
        ),
        "temporal_similarity": CFG.ocr_temporal_similarity,
        "probe_interval_seconds": CFG.ocr_probe_interval_seconds,
        "maximum_probe_frames": CFG.ocr_probe_max_frames,
        "edge_offset_seconds": CFG.ocr_probe_edge_offset_seconds,
    },
    "keyword_configuration": {
        "ngram_range": [1, 2],
        "stop_words": "english",
        "use_mmr": True,
        "diversity": 0.7,
        "top_n": 8,
    },
    "whisper_model": CFG.whisper_model,
    "vqa_model": CFG.vqa_model,
    "clip_model": CFG.clip_model,
    "semantic_model": CFG.semantic_model,
    "software_versions": {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "openai_whisper": importlib_metadata.version("openai-whisper"),
        "transformers": importlib_metadata.version("transformers"),
        "sentence_transformers": importlib_metadata.version("sentence-transformers"),
        "keybert": importlib_metadata.version("keybert"),
        "jiwer": importlib_metadata.version("jiwer"),
        "pytesseract": importlib_metadata.version("pytesseract"),
        "scenedetect": importlib_metadata.version("scenedetect"),
        "jsonschema": importlib_metadata.version("jsonschema"),
    },
    "resolved_model_revisions": {
        "vqa_model": resolved_model_revision(MODELS["vqa_model"]),
        "clip_model": resolved_model_revision(MODELS["clip_model"]),
        "semantic_model": resolved_model_revision(MODELS["semantic_encoder"]),
    },
}
PIPELINE_SIGNATURE = hashlib.sha256(
    json.dumps(PIPELINE_CONFIG, sort_keys=True).encode("utf-8")
).hexdigest()[:12]
PREDICTION_FILE = CFG.artifact_dir / f"predictions_{PIPELINE_SIGNATURE}.json"
FLAT_PREDICTION_FILE = CFG.artifact_dir / f"predictions_{PIPELINE_SIGNATURE}.csv"


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def write_json_atomic(payload: dict[str, Any], path: Path) -> None:
    temp_path = path.with_name(path.stem + ".partial.json")
    with open(temp_path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
    os.replace(temp_path, path)


def load_existing_predictions() -> dict[str, dict[str, Any]]:
    if not PREDICTION_FILE.exists():
        return {}
    with open(PREDICTION_FILE, encoding="utf-8") as handle:
        payload = json.load(handle)
    if payload.get("pipeline_signature") != PIPELINE_SIGNATURE:
        return {}
    return {
        record["clip_id"]: record
        for record in payload.get("records", [])
    }


def save_prediction_records(records: dict[str, dict[str, Any]]) -> None:
    ordered = [
        records[clip_id]
        for clip_id in MANIFEST["clip_id"]
        if clip_id in records
    ]
    payload = {
        "schema_version": "4.0",
        "pipeline_signature": PIPELINE_SIGNATURE,
        "pipeline_config": PIPELINE_CONFIG,
        "updated_utc": utc_now(),
        "records": ordered,
    }
    write_json_atomic(payload, PREDICTION_FILE)


def flatten_prediction_records(records: Iterable[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for record in records:
        base = {
            "clip_id": record["clip_id"],
            "source_id": record["source_id"],
            "source_video": record["source_video"],
            "split": record["split"],
            "status": record["status"],
            "processing_seconds": record.get("processing_seconds"),
        }
        if record["status"] != "ok":
            rows.append({**base, "variant": "error"})
            continue
        common = record["predictions"]["common"]
        for variant in ("single_frame", "multi_frame"):
            visual = record["predictions"][variant]
            rows.append({
                **base,
                "variant": variant,
                "transcript": common["transcript"],
                "keywords": "; ".join(common["keywords"]),
                "on_screen_text": " ".join(visual["on_screen_text"]),
                "visual_tags": "; ".join(visual["visual_tags"]),
                "people_count": visual["people_count_numeric"],
            })
    return pd.DataFrame(rows)


def infer_ocr_probe_frame(image: Image.Image) -> dict[str, Any]:
    text_lines, text_confidences = ocr_image(image)
    return {
        "on_screen_text": text_lines,
        "ocr_confidences": text_confidences,
    }


def make_clip_error_record(
    row: Any,
    exception: Exception,
    started: float,
) -> dict[str, Any]:
    return {
        "clip_id": row.clip_id,
        "source_id": row.source_id,
        "source_video": row.source_video,
        "split": row.split,
        "clip_index": int(row.clip_index),
        "start_sec": float(row.start_sec),
        "end_sec": float(row.end_sec),
        "clip_path": str(row.clip_path),
        "status": "error",
        "error": {
            "type": type(exception).__name__,
            "message": str(exception),
            "traceback": traceback.format_exc(limit=8),
        },
        "pipeline_signature": PIPELINE_SIGNATURE,
        "processed_utc": utc_now(),
        "processing_seconds": round(time.perf_counter() - started, 3),
    }


def process_single_clip_parallel(
    row: Any,
    ocr_executor: ThreadPoolExecutor,
) -> dict[str, Any]:
    started = time.perf_counter()
    ocr_futures = []
    try:
        persistent_clip = Path(row.clip_path)
        local_clip = copy_clip_to_local(persistent_clip)
        technical = probe_media(local_clip)
        frames = extract_keyframes(
            local_clip,
            row.clip_id,
            CFG.frame_fractions,
        )
        frame_selection = dict(FRAME_SELECTION_CACHE[row.clip_id])
        frame_times = [
            float(value)
            for value in frame_selection["frame_times_sec"]
        ]
        duration = float(frame_selection["duration_sec"])
        middle_index = min(
            range(len(frame_times)),
            key=lambda index: abs(frame_times[index] - duration * 0.5),
        )

        # Submit CPU OCR first.  These subprocess-backed tasks run while
        # this or another clip waits for/uses the shared GPU models.
        ocr_started = time.perf_counter()
        ocr_frames, ocr_times = extract_ocr_probe_frames(
            local_clip,
            row.clip_id,
            frame_times,
        )
        dense_ocr_futures = [
            ocr_executor.submit(
                infer_ocr_probe_frame,
                image.copy(),
            )
            for image in ocr_frames
        ]
        ocr_futures.extend(dense_ocr_futures)

        centre_time = frame_times[middle_index]
        nearest_dense_index = min(
            range(len(ocr_times)),
            key=lambda index: abs(float(ocr_times[index]) - centre_time),
        )
        if abs(float(ocr_times[nearest_dense_index]) - centre_time) <= 0.08:
            centre_ocr_future = dense_ocr_futures[nearest_dense_index]
        else:
            centre_ocr_future = ocr_executor.submit(
                infer_ocr_probe_frame,
                frames[middle_index].copy(),
            )
            ocr_futures.append(centre_ocr_future)

        queue_started = time.perf_counter()
        with MODEL_INFERENCE_LOCK:
            model_queue_wait_seconds = time.perf_counter() - queue_started
            asr_started = time.perf_counter()
            asr = transcribe_clip(local_clip, technical["has_audio"])
            keywords = extract_keywords(asr["text"])
            asr_keyword_processing_seconds = (
                time.perf_counter() - asr_started
            )

            visual_started = time.perf_counter()
            visual_outputs = [
                infer_frame_visual_only(image)
                for image in frames
            ]
            visual_processing_seconds = (
                time.perf_counter() - visual_started
            )

        centre_ocr = centre_ocr_future.result()
        dense_ocr_outputs = [
            future.result() for future in dense_ocr_futures
        ]
        ocr_processing_seconds = time.perf_counter() - ocr_started

        centre_output = {
            **visual_outputs[middle_index],
            **centre_ocr,
        }
        visual_only_outputs = [
            {
                **output,
                "on_screen_text": [],
                "ocr_confidences": [],
            }
            for output in visual_outputs
        ]
        single = aggregate_frame_outputs([centre_output])
        multi = aggregate_frame_outputs(visual_only_outputs)
        dense_ocr_lines, dense_ocr_evidence = aggregate_temporal_ocr(
            dense_ocr_outputs
        )
        multi["on_screen_text"] = dense_ocr_lines
        multi["ocr_temporal_evidence"] = dense_ocr_evidence
        multi["ocr_frame_times_sec"] = [
            round(float(value), 4) for value in ocr_times
        ]

        return {
            "clip_id": row.clip_id,
            "source_id": row.source_id,
            "source_video": row.source_video,
            "split": row.split,
            "clip_index": int(row.clip_index),
            "start_sec": float(row.start_sec),
            "end_sec": float(row.end_sec),
            "clip_path": str(persistent_clip),
            "status": "ok",
            "error": None,
            "pipeline_signature": PIPELINE_SIGNATURE,
            "processed_utc": utc_now(),
            "processing_seconds": round(
                time.perf_counter() - started, 3
            ),
            "model_queue_wait_seconds": round(
                model_queue_wait_seconds, 3
            ),
            "asr_keyword_processing_seconds": round(
                asr_keyword_processing_seconds, 3
            ),
            "visual_processing_seconds": round(
                visual_processing_seconds, 3
            ),
            "ocr_processing_seconds": round(
                ocr_processing_seconds, 3
            ),
            "frame_sampling": frame_selection,
            "ocr_sampling": {
                "policy": "dense_uniform_plus_scene_frames",
                "frame_times_sec": [
                    round(float(value), 4) for value in ocr_times
                ],
                "frame_count": len(ocr_times),
                "interval_seconds": float(
                    CFG.ocr_probe_interval_seconds
                ),
                "maximum_frames": int(CFG.ocr_probe_max_frames),
            },
            "parallel_execution": {
                "enabled": bool(CFG.parallel_metadata_processing),
                "shared_model_inference_serialized": True,
            },
            "source_sha256": row.source_sha256,
            "clip_sha256": row.clip_sha256,
            "technical": technical,
            "predictions": {
                "common": {
                    "transcript": asr["text"],
                    "transcript_segments": asr["segments"],
                    "language": asr["language"],
                    "keywords": keywords,
                },
                "single_frame": single,
                "multi_frame": multi,
            },
        }
    except Exception as exc:
        for future in ocr_futures:
            future.cancel()
        return make_clip_error_record(row, exc, started)


def process_all_clips() -> dict[str, dict[str, Any]]:
    records = load_existing_predictions()
    pending_rows = []
    for row in MANIFEST.itertuples(index=False):
        previous = records.get(row.clip_id)
        if (
            previous
            and previous.get("status") == "ok"
            and previous.get("pipeline_signature") == PIPELINE_SIGNATURE
            and not CFG.force_reprocess
        ):
            continue
        pending_rows.append(row)

    available_cpus = max(1, int(os.cpu_count() or 1))
    if CFG.parallel_metadata_processing:
        clip_workers = min(
            max(1, int(CFG.pipeline_clip_workers)),
            available_cpus,
            max(1, len(pending_rows)),
        )
        ocr_workers = min(
            max(1, int(CFG.ocr_parallel_workers)),
            available_cpus,
        )
    else:
        clip_workers = 1
        ocr_workers = 1
    checkpoint_every = max(1, int(CFG.checkpoint_every_clips))

    print(
        "Metadata execution:",
        f"{clip_workers} clip worker(s),",
        f"{ocr_workers} OCR worker(s),",
        "shared model inference serialized",
    )
    failures = []
    completed_since_checkpoint = 0

    if pending_rows:
        with ThreadPoolExecutor(
            max_workers=ocr_workers,
            thread_name_prefix="ocr",
        ) as ocr_executor:
            with ThreadPoolExecutor(
                max_workers=clip_workers,
                thread_name_prefix="clip",
            ) as clip_executor:
                future_to_row = {
                    clip_executor.submit(
                        process_single_clip_parallel,
                        row,
                        ocr_executor,
                    ): row
                    for row in pending_rows
                }
                progress = tqdm(
                    total=len(future_to_row),
                    desc="Metadata inference",
                )
                try:
                    for future in as_completed(future_to_row):
                        row = future_to_row[future]
                        record = future.result()
                        records[row.clip_id] = record
                        completed_since_checkpoint += 1

                        if record["status"] == "error":
                            failures.append(row.clip_id)
                            if CFG.fail_fast:
                                save_prediction_records(records)
                                for pending in future_to_row:
                                    pending.cancel()
                                error = record.get("error", {})
                                raise RuntimeError(
                                    f"{row.clip_id} failed: "
                                    f"{error.get('type')}: "
                                    f"{error.get('message')}"
                                )

                        if completed_since_checkpoint >= checkpoint_every:
                            save_prediction_records(records)
                            completed_since_checkpoint = 0
                        progress.update(1)
                finally:
                    progress.close()

    save_prediction_records(records)
    ordered_records = [
        records[clip_id]
        for clip_id in MANIFEST["clip_id"]
        if clip_id in records
    ]
    flat = flatten_prediction_records(ordered_records)
    write_csv_atomic(flat, FLAT_PREDICTION_FILE)
    print(f"Predictions: {PREDICTION_FILE}")
    print(f"Flat export: {FLAT_PREDICTION_FILE}")
    print(
        "Successful clips:",
        sum(record["status"] == "ok" for record in records.values()),
    )
    print(
        "Failed clips:",
        sum(record["status"] == "error" for record in records.values()),
    )
    if failures:
        print("Failures in this run:", failures)
    return records


In [16]:
CFG.force_reprocess = False

In [17]:
RECORDS = process_all_clips()

Metadata execution: 1 clip worker(s), 2 OCR worker(s), shared model inference serialized
Predictions: /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4/predictions_7ed66532a36a.json
Flat export: /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4/predictions_7ed66532a36a.csv
Successful clips: 626
Failed clips: 0


## 10. Inspect one clip and its predictions

In [18]:
def review_clip(clip_id: Optional[str] = None) -> None:
    available = [
        clip_id_
        for clip_id_, record in RECORDS.items()
        if record.get("status") == "ok"
    ]
    if not available:
        print("No successful predictions are available.")
        return
    selected = clip_id or available[0]
    if selected not in RECORDS:
        raise KeyError(f"Unknown clip_id: {selected}")
    record = RECORDS[selected]
    local_clip = copy_clip_to_local(Path(record["clip_path"]))
    preview_dir = LOCAL_CACHE / "previews"
    preview_dir.mkdir(parents=True, exist_ok=True)
    preview = preview_dir / f"{selected}__preview.mp4"
    if not preview.exists() or preview.stat().st_size == 0:
        partial = preview.with_name(preview.stem + ".partial.mp4")
        partial.unlink(missing_ok=True)
        run_checked([
            "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
            "-i", str(local_clip),
            "-vf", "scale=640:-2",
            "-c:v", "libx264", "-preset", "veryfast", "-crf", "28",
            "-c:a", "aac", "-b:a", "64k",
            "-movflags", "+faststart",
            str(partial),
        ])
        os.replace(partial, preview)
    display(Video(str(preview), embed=True, width=720))
    print(json.dumps(record["predictions"], indent=2, ensure_ascii=False))

print("Call review_clip() or review_clip('a_clip_id') when you want a preview.")

Call review_clip() or review_clip('a_clip_id') when you want a preview.


## 11. Operational definition of automatic ground truth

The project requirement excludes manual annotation of the NVTV material. This notebook therefore uses the following operational definition:

**Automatic ground truth = technical metadata read directly from the media container plus descriptive metadata selected from heterogeneous pretrained-model families using deterministic evidence aggregation.**

The JSON header identifies the result as an automated multi-model silver standard. Without an independent reference, semantic correctness is not directly measurable; disagreement, missing evidence and model dependence are retained rather than hidden. Gemini is an optional model-family vote, not an authority.

| Field | Evidence families | Selection and agreement rule |
|---|---|---|
| Transcript | Whisper-small, Whisper-turbo | Turbo deterministic candidate; inter-model WER converted to agreement |
| On-screen text | Tesseract, EasyOCR; Gemini diagnostic | Publish only temporally persistent fuzzy matches supported by both OCR engines; Gemini cannot override |
| Keywords | KeyBERT, YAKE, TF-IDF | Weighted ranked-set consensus |
| Visual tags | CLIP ViT-B/32, CLIP ViT-L/14, optional Gemini | Closed vocabulary; at least two model families must support a published tag |
| People count | BLIP VQA, ViLT VQA, DETR, optional Gemini | One numeric vote per family; mode with median tie-break |

Model-family weights are equal by default. They may be changed only using independently measured benchmark reliability, with the weight file retained as provenance. Centre-frame outputs are diagnostics and never receive extra votes.


## 11A. Frame-sampling ablation on the frozen calibration subset

This experiment compares the centre-frame baseline, fixed 20/50/80% sampling and the proposed scene-aware policy. It does not claim accuracy. It measures coverage, change from the centre baseline and cross-frame stability. Only the calibration subset is used to avoid selecting the final policy after inspecting evaluation clips.

In [19]:
def ablation_set_f1(first: Iterable[str], second: Iterable[str]) -> float:
    first_set = {
        key for value in first
        if (key := normalize_key(value))
    }
    second_set = {
        key for value in second
        if (key := normalize_key(value))
    }
    if not first_set and not second_set:
        return np.nan
    if not first_set or not second_set:
        return 0.0
    overlap = len(first_set & second_set)
    return float(2.0 * overlap / (len(first_set) + len(second_set)))


def ablation_semantic_similarity(first: str, second: str) -> float:
    first, second = normalize_space(first), normalize_space(second)
    if not first or not second:
        return np.nan
    embeddings = MODELS["semantic_encoder"].encode(
        [first, second],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return float(np.clip(embeddings[0] @ embeddings[1], -1.0, 1.0))


def visual_metadata_coverage(value: dict[str, Any]) -> float:
    fields = (
        "on_screen_text", "visual_tags", "people_count_numeric",
    )
    present = []
    for field in fields:
        item = value.get(field)
        if field == "people_count_numeric":
            present.append(item is not None)
        elif isinstance(item, (list, tuple, set, dict)):
            present.append(bool(item))
        else:
            present.append(bool(normalize_space(item)))
    return float(np.mean(present))


def compare_visual_variants(
    baseline: dict[str, Any],
    candidate: dict[str, Any],
) -> dict[str, float]:
    measures = {
        "ocr_set_f1_vs_centre": ablation_set_f1(
            baseline["on_screen_text"], candidate["on_screen_text"]
        ),
        "tag_set_f1_vs_centre": ablation_set_f1(
            baseline["visual_tags"], candidate["visual_tags"]
        ),
    }
    first_count = baseline.get("people_count_numeric")
    second_count = candidate.get("people_count_numeric")
    measures["people_count_stability_vs_centre"] = (
        float(np.exp(-abs(first_count - second_count)))
        if first_count is not None and second_count is not None
        else np.nan
    )
    available = [
        value for value in measures.values() if np.isfinite(value)
    ]
    measures["mean_stability_vs_centre"] = (
        float(np.mean(available)) if available else np.nan
    )
    return measures


def run_frame_sampling_ablation() -> tuple[pd.DataFrame, pd.DataFrame]:
    candidates = MANIFEST[MANIFEST["split"].eq("calibration")]
    if candidates.empty:
        candidates = MANIFEST
    candidates = candidates.head(CFG.ablation_max_clips)
    rows = []

    for row in tqdm(
        candidates.itertuples(index=False),
        total=len(candidates),
        desc="Frame ablation",
    ):
        record = RECORDS.get(row.clip_id)
        if not record or record.get("status") != "ok":
            continue
        baseline = record["predictions"]["single_frame"]
        scene_aware = record["predictions"]["multi_frame"]
        duration = float(record["technical"]["duration_sec"])
        local_clip = copy_clip_to_local(Path(row.clip_path))
        fixed_times, _ = choose_frame_times(
            local_clip, duration, "fixed_three"
        )
        fixed_images = extract_keyframes_at_times(
            local_clip,
            row.clip_id,
            fixed_times,
            cache_tag="ablation_fixed_three",
        )
        started = time.perf_counter()
        fixed_outputs = [infer_frame(image) for image in fixed_images]
        fixed_three = aggregate_frame_outputs(fixed_outputs)
        fixed_runtime = time.perf_counter() - started

        variants = {
            "centre_frame": baseline,
            "fixed_three": fixed_three,
            "scene_aware": scene_aware,
        }
        for variant, metadata in variants.items():
            comparison = (
                {
                    "ocr_set_f1_vs_centre": 1.0,
                    "tag_set_f1_vs_centre": 1.0,
                    "people_count_stability_vs_centre": 1.0,
                    "mean_stability_vs_centre": 1.0,
                }
                if variant == "centre_frame"
                else compare_visual_variants(baseline, metadata)
            )
            rows.append({
                "clip_id": row.clip_id,
                "source_id": row.source_id,
                "variant": variant,
                "frame_count": (
                    1 if variant == "centre_frame"
                    else len(fixed_times) if variant == "fixed_three"
                    else record["frame_sampling"]["frame_count"]
                ),
                "visual_coverage": visual_metadata_coverage(metadata),
                "ocr_item_count": len(metadata["on_screen_text"]),
                "extra_visual_runtime_seconds": (
                    fixed_runtime if variant == "fixed_three" else np.nan
                ),
                **comparison,
            })

    detail = pd.DataFrame(rows)
    if detail.empty:
        summary = pd.DataFrame()
    else:
        numeric = [
            column for column in detail.select_dtypes(include=[np.number])
            if column not in {"frame_count"}
        ]
        summary = (
            detail.groupby("variant", as_index=False)
            .agg(
                n_clips=("clip_id", "nunique"),
                mean_frame_count=("frame_count", "mean"),
                **{
                    f"mean_{column}": (column, "mean")
                    for column in numeric
                },
            )
        )
    write_csv_atomic(
        detail,
        CFG.artifact_dir / "frame_sampling_ablation_detail.csv",
    )
    write_csv_atomic(
        summary,
        CFG.artifact_dir / "frame_sampling_ablation_summary.csv",
    )
    display(summary.round(4))
    return detail, summary


FRAME_ABLATION_DETAIL, FRAME_ABLATION_SUMMARY = (
    run_frame_sampling_ablation()
)

Frame ablation:   0%|          | 0/8 [00:00<?, ?it/s]

,variant,n_clips,mean_frame_count,mean_visual_coverage,mean_ocr_item_count,mean_extra_visual_runtime_seconds,mean_ocr_set_f1_vs_centre,mean_tag_set_f1_vs_centre,mean_people_count_stability_vs_centre,mean_mean_stability_vs_centre
0,centre_frame,8,1.000,1.0,14.5,NaN,1.0000,1.000,1.0000,1.0000
1,fixed_three,8,3.000,1.0,3.0,11.8682,0.3027,0.650,0.6879,0.5469
2,scene_aware,8,4.375,1.0,23.0,NaN,0.3115,0.675,0.6089,0.5318


In [20]:
VERIFIER_CONFIG = {
    "schema_version": "4.0",
    "seed": SEED,
    "human_annotation": False,
    "whisper_model": CFG.verifier_whisper_model,
    "ocr_model": "EasyOCR 1.7.2 English",
    "easyocr_min_confidence": CFG.easyocr_min_confidence,
    "ocr_preprocessing": ["CLAHE", "unsharp mask", "Otsu threshold"],
    "ocr_minimum_frame_occurrences": CFG.ocr_min_frame_occurrences,
    "ocr_temporal_similarity": CFG.ocr_temporal_similarity,
    "ocr_cross_engine_similarity": CFG.ocr_cross_engine_similarity,
    "keyword_models": ["YAKE 0.6.0", "TF-IDF"],
    "vqa_model": CFG.verifier_vqa_model,
    "clip_model": CFG.verifier_clip_model,
    "people_model": CFG.verifier_people_model,
    "frame_fractions": list(CFG.frame_fractions),
    "frame_sampling_policy": CFG.frame_sampling_policy,
    "max_scene_frames": CFG.max_scene_frames,
    "visual_labels": list(VISUAL_LABELS),
    "software_versions": {
        "easyocr": importlib_metadata.version("easyocr"),
        "yake": importlib_metadata.version("yake"),
        "transformers": importlib_metadata.version("transformers"),
        "sentence_transformers": importlib_metadata.version(
            "sentence-transformers"
        ),
    },
}
VERIFIER_SIGNATURE = hashlib.sha256(
    json.dumps(VERIFIER_CONFIG, sort_keys=True).encode("utf-8")
).hexdigest()[:12]
VERIFIER_FILE = (
    CFG.artifact_dir
    / f"verifier_predictions_{VERIFIER_SIGNATURE}.json"
)
GROUND_TRUTH_FILE = CFG.artifact_dir / "ground_truth_metadata.json"
GROUND_TRUTH_CSV = CFG.artifact_dir / "ground_truth_metadata.csv"


def load_verifier_records() -> dict[str, dict[str, Any]]:
    if not VERIFIER_FILE.exists():
        return {}
    with open(VERIFIER_FILE, encoding="utf-8") as handle:
        payload = json.load(handle)
    if payload.get("verifier_signature") != VERIFIER_SIGNATURE:
        return {}
    return {
        record["clip_id"]: record
        for record in payload.get("records", [])
    }


def save_verifier_records(records: dict[str, dict[str, Any]]) -> None:
    ordered = [
        records[clip_id]
        for clip_id in MANIFEST["clip_id"]
        if clip_id in records
    ]
    write_json_atomic(
        {
            "schema_version": "4.0",
            "verifier_signature": VERIFIER_SIGNATURE,
            "verifier_config": VERIFIER_CONFIG,
            "updated_utc": utc_now(),
            "records": ordered,
        },
        VERIFIER_FILE,
    )


def release_primary_models() -> None:
    global MODELS
    if isinstance(MODELS, dict):
        MODELS.clear()
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print("Primary models released before verifier loading.")


release_primary_models()

Primary models released before verifier loading.


## 12. Verifier phase A: independent text evidence

Whisper-turbo regenerates transcripts. YAKE and corpus-level TF-IDF provide keyword candidates distinct from KeyBERT.

In [21]:
import yake
from sklearn.feature_extraction.text import TfidfVectorizer


def top_tfidf_terms(
    transcripts: dict[str, str],
    top_n: int = 8,
) -> dict[str, list[str]]:
    nonempty_ids = [
        clip_id
        for clip_id, text in transcripts.items()
        if normalize_space(text)
    ]
    output = {clip_id: [] for clip_id in transcripts}
    if not nonempty_ids:
        return output
    documents = [transcripts[clip_id] for clip_id in nonempty_ids]
    try:
        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            max_features=5000,
        )
        matrix = vectorizer.fit_transform(documents)
    except ValueError:
        return output
    terms = np.asarray(vectorizer.get_feature_names_out())
    for row_index, clip_id in enumerate(nonempty_ids):
        values = matrix.getrow(row_index).toarray().ravel()
        ranked = np.argsort(values)[::-1]
        output[clip_id] = [
            str(terms[index])
            for index in ranked
            if values[index] > 0
        ][:top_n]
    return output


def process_verifier_text() -> dict[str, dict[str, Any]]:
    records = load_verifier_records()
    asr_model_b = whisper.load_model(
        CFG.verifier_whisper_model,
        device=DEVICE,
    )
    yake_extractor = yake.KeywordExtractor(
        lan="en",
        n=2,
        dedupLim=0.8,
        top=8,
    )

    for row in tqdm(
        MANIFEST.itertuples(index=False),
        total=len(MANIFEST),
        desc="Verifier ASR",
    ):
        primary = RECORDS.get(row.clip_id)
        if not primary or primary.get("status") != "ok":
            continue
        existing = records.get(row.clip_id, {})
        if (
            existing.get("text_status") == "ok"
            and not CFG.force_reprocess
        ):
            continue

        started = time.perf_counter()
        try:
            local_clip = copy_clip_to_local(Path(row.clip_path))
            if primary["technical"].get("has_audio"):
                result = asr_model_b.transcribe(
                    str(local_clip),
                    task="transcribe",
                    fp16=(DEVICE == "cuda"),
                    temperature=0.0,
                    condition_on_previous_text=False,
                    verbose=False,
                )
                transcript_b = normalize_space(result.get("text", ""))
            else:
                transcript_b = ""

            if transcript_b:
                keywords_yake = [
                    normalize_space(keyword)
                    for keyword, _ in yake_extractor.extract_keywords(
                        transcript_b
                    )
                ]
            else:
                keywords_yake = []

            existing.update({
                "clip_id": row.clip_id,
                "source_id": row.source_id,
                "source_video": row.source_video,
                "text_status": "ok",
                "transcript_whisper_turbo": transcript_b,
                "keywords_yake": keywords_yake,
                "text_processing_seconds": round(
                    time.perf_counter() - started,
                    3,
                ),
            })
        except Exception as exc:
            existing.update({
                "clip_id": row.clip_id,
                "source_id": row.source_id,
                "source_video": row.source_video,
                "text_status": "error",
                "text_error": {
                    "type": type(exc).__name__,
                    "message": str(exc),
                    "traceback": traceback.format_exc(limit=8),
                },
            })
            if CFG.fail_fast:
                records[row.clip_id] = existing
                save_verifier_records(records)
                raise
        records[row.clip_id] = existing
        save_verifier_records(records)

    transcript_map = {
        clip_id: record.get("transcript_whisper_turbo", "")
        for clip_id, record in records.items()
        if record.get("text_status") == "ok"
    }
    tfidf_map = top_tfidf_terms(transcript_map)
    for clip_id, keywords in tfidf_map.items():
        records[clip_id]["keywords_tfidf"] = keywords
    save_verifier_records(records)

    del asr_model_b
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print("Verifier text phase complete.")
    return records


VERIFIER_RECORDS = process_verifier_text()

100%|█████████████████████████████████████| 1.51G/1.51G [00:24<00:00, 67.3MiB/s]


Verifier ASR:   0%|          | 0/626 [00:00<?, ?it/s]

Verifier text phase complete.


## 13. Verifier phase B: independent visual evidence

The visual verifier uses EasyOCR, CLIP ViT-L/14, ViLT VQA and DETR person detection. These models are loaded only after the primary and verifier ASR models are released to control GPU memory.

In [22]:
import easyocr
from sentence_transformers import SentenceTransformer
from transformers import (
    DetrForObjectDetection,
    DetrImageProcessor,
    ViltForQuestionAnswering,
    ViltProcessor,
)


def move_batch_with_dtype(
    batch: Any,
    floating_dtype: torch.dtype,
) -> dict[str, torch.Tensor]:
    moved = {}
    for key, value in batch.items():
        if torch.is_tensor(value):
            if value.is_floating_point():
                moved[key] = value.to(
                    device=DEVICE,
                    dtype=floating_dtype,
                )
            else:
                moved[key] = value.to(device=DEVICE)
        else:
            moved[key] = value
    return moved


def load_visual_verifier_models() -> dict[str, Any]:
    print("Loading EasyOCR...")
    easyocr_reader = easyocr.Reader(
        ["en"],
        gpu=(DEVICE == "cuda"),
        verbose=False,
    )

    print("Loading verifier CLIP:", CFG.verifier_clip_model)
    clip_processor_b = CLIPProcessor.from_pretrained(
        CFG.verifier_clip_model
    )
    clip_model_b = CLIPModel.from_pretrained(
        CFG.verifier_clip_model,
        torch_dtype=MODEL_DTYPE,
    ).to(DEVICE).eval()

    print("Loading ViLT VQA:", CFG.verifier_vqa_model)
    vilt_processor = ViltProcessor.from_pretrained(
        CFG.verifier_vqa_model
    )
    vilt_model = ViltForQuestionAnswering.from_pretrained(
        CFG.verifier_vqa_model
    ).to(DEVICE).eval()

    print("Loading DETR:", CFG.verifier_people_model)
    detr_processor = DetrImageProcessor.from_pretrained(
        CFG.verifier_people_model,
        revision="no_timm",
    )
    detr_model = DetrForObjectDetection.from_pretrained(
        CFG.verifier_people_model,
        revision="no_timm",
    ).to(DEVICE).eval()

    semantic_encoder = SentenceTransformer(
        CFG.semantic_model,
        device=DEVICE,
    )

    label_inputs = clip_processor_b(
        text=[f"a photograph of {label}" for label in VISUAL_LABELS],
        return_tensors="pt",
        padding=True,
    )
    with torch.inference_mode():
        text_features = tensor_output(
            clip_model_b.get_text_features(
                **move_batch_with_dtype(label_inputs, MODEL_DTYPE)
            )
        )
        text_features = torch.nn.functional.normalize(
            text_features,
            dim=-1,
        )

    return {
        "easyocr": easyocr_reader,
        "clip_processor": clip_processor_b,
        "clip_model": clip_model_b,
        "clip_text_features": text_features,
        "vilt_processor": vilt_processor,
        "vilt_model": vilt_model,
        "detr_processor": detr_processor,
        "detr_model": detr_model,
        "semantic_encoder": semantic_encoder,
    }


VERIFIER_MODELS = load_visual_verifier_models()
VERIFIER_RESOLVED_REVISIONS = {
    "clip_model": resolved_model_revision(VERIFIER_MODELS["clip_model"]),
    "vqa_model": resolved_model_revision(VERIFIER_MODELS["vilt_model"]),
    "people_model": resolved_model_revision(VERIFIER_MODELS["detr_model"]),
    "semantic_model": resolved_model_revision(VERIFIER_MODELS["semantic_encoder"]),
}
display(pd.DataFrame(
    [{"component": key, "resolved_revision": value}
     for key, value in VERIFIER_RESOLVED_REVISIONS.items()]
))

Loading EasyOCR...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading verifier CLIP: openai/clip-vit-large-patch14


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,component,resolved_revision
0,clip_model,32bd64288804d66eefd0ccbe215aa642df71cc41
1,vqa_model,d0a1f6ab88522427a7ae76ceb6e1e1e7b68a1d08
2,people_model,70120ba84d68ca1211e007c4fb61d0cd5424be54
3,semantic_model,1110a243fdf4706b3f48f1d95db1a4f5529b4d41


In [23]:
def easyocr_image(
    image: Image.Image,
) -> tuple[list[str], list[float]]:
    variants = prepare_ocr_variants(image)

    if not isinstance(variants, dict) or not variants:
        raise ValueError(
            "prepare_ocr_variants() must return a non-empty dictionary."
        )

    # Supports names such as enhanced, CLAHE, adaptive_threshold,
    # Otsu, binary, grayscale, original, etc.
    preferred_terms = (
        "enhanced",
        "clahe",
        "contrast",
        "threshold",
        "adaptive",
        "otsu",
        "binary",
        "grayscale",
        "greyscale",
        "gray",
        "original",
    )

    selected_names: list[str] = []

    for term in preferred_terms:
        for name in variants:
            if (
                term in str(name).lower()
                and name not in selected_names
            ):
                selected_names.append(name)

            if len(selected_names) == 2:
                break

        if len(selected_names) == 2:
            break

    # Fall back to any available variants.
    if len(selected_names) < 2:
        for name in variants:
            if name not in selected_names:
                selected_names.append(name)

            if len(selected_names) == 2:
                break

    lines: list[str] = []
    confidences: list[float] = []

    for variant_name in selected_names:
        variant = variants[variant_name]

        if isinstance(variant, Image.Image):
            variant_array = np.asarray(variant)
        else:
            variant_array = np.asarray(variant)

        if variant_array.size == 0:
            continue

        results = VERIFIER_MODELS["easyocr"].readtext(
            variant_array,
            detail=1,
            paragraph=False,
            batch_size=8,
            workers=0,
        )

        for result in results:
            if len(result) != 3:
                continue

            _, text, confidence = result
            clean = normalize_space(text)
            confidence = float(confidence)

            if (
                clean
                and confidence >= CFG.easyocr_min_confidence
                and plausible_ocr_text(clean)
            ):
                lines.append(clean)
                confidences.append(confidence)

    if not lines:
        return [], []

    merged_lines, evidence = merge_similar_ocr_lines(
        lines,
        confidences,
    )

    merged_confidences = [
        float(item.get("mean_confidence", 0.0))
        for item in evidence
    ]

    return merged_lines, merged_confidences



@torch.inference_mode()
def vilt_answer(image: Image.Image, question: str) -> str:
    inputs = VERIFIER_MODELS["vilt_processor"](
        image,
        question,
        return_tensors="pt",
    )
    outputs = VERIFIER_MODELS["vilt_model"](
        **move_batch_with_dtype(inputs, torch.float32)
    )
    index = int(outputs.logits.argmax(-1).item())
    return normalize_space(
        VERIFIER_MODELS["vilt_model"].config.id2label[index]
    )


@torch.inference_mode()
def verifier_clip_scores(image: Image.Image) -> dict[str, float]:
    inputs = VERIFIER_MODELS["clip_processor"](
        images=image,
        return_tensors="pt",
    )
    image_features = tensor_output(
        VERIFIER_MODELS["clip_model"].get_image_features(
            **move_batch_with_dtype(inputs, MODEL_DTYPE)
        )
    )
    image_features = torch.nn.functional.normalize(
        image_features,
        dim=-1,
    )
    similarities = (
        image_features
        @ VERIFIER_MODELS["clip_text_features"].T
    ).squeeze(0)
    relative_scores = torch.softmax(
        similarities * 100.0,
        dim=-1,
    )
    return {
        label: float(score)
        for label, score in zip(
            VISUAL_LABELS,
            relative_scores.cpu(),
        )
    }


@torch.inference_mode()
def detr_people_count(
    image: Image.Image,
    threshold: float = 0.70,
) -> int:
    inputs = VERIFIER_MODELS["detr_processor"](
        images=image,
        return_tensors="pt",
    )
    outputs = VERIFIER_MODELS["detr_model"](
        **move_batch_with_dtype(inputs, torch.float32)
    )
    target_sizes = torch.tensor(
        [image.size[::-1]],
        device=DEVICE,
    )
    result = VERIFIER_MODELS["detr_processor"].post_process_object_detection(
        outputs,
        target_sizes=target_sizes,
        threshold=threshold,
    )[0]
    labels = [
        VERIFIER_MODELS["detr_model"].config.id2label[
            int(label.item())
        ].lower()
        for label in result["labels"]
    ]
    return sum(label == "person" for label in labels)


def infer_verifier_frame(image: Image.Image) -> dict[str, Any]:
    text_lines, text_confidences = easyocr_image(image)
    return {
        "on_screen_text": text_lines,
        "ocr_confidences": text_confidences,
        "visual_tag_scores": verifier_clip_scores(image),
        "facts": {
            field: vilt_answer(image, question)
            for field, question in VQA_QUESTIONS.items()
        },
        "detr_people_count": detr_people_count(image),
    }


def summarize_verifier_frames(
    frame_outputs: list[dict[str, Any]],
    frame_times: list[float],
    duration: float,
) -> dict[str, Any]:
    if not frame_outputs:
        raise ValueError("At least one verifier frame is required.")
    if len(frame_outputs) != len(frame_times):
        raise ValueError("Frame outputs and timestamps must align.")
    centre_index = min(
        range(len(frame_times)),
        key=lambda index: abs(frame_times[index] - duration * 0.5),
    )
    mean_scores = {
        label: float(np.mean([
            output["visual_tag_scores"][label]
            for output in frame_outputs
        ]))
        for label in VISUAL_LABELS
    }
    centre = frame_outputs[centre_index]
    ocr_lines, ocr_temporal_evidence = aggregate_temporal_ocr(
        frame_outputs
    )
    return {
        "single_frame": {
            "on_screen_text": centre["on_screen_text"],
            "ocr_confidences": centre["ocr_confidences"],
            "visual_tags": top_tags(
                centre["visual_tag_scores"],
                CFG.tag_top_k,
            ),
            "facts": centre["facts"],
            "detr_people_count": centre["detr_people_count"],
        },
        "multi_frame": {
            "frame_times_sec": [round(float(value), 4) for value in frame_times],
            "on_screen_text": ocr_lines,
            "ocr_temporal_evidence": ocr_temporal_evidence,
            "visual_tags": top_tags(mean_scores, CFG.tag_top_k),
            "facts_per_frame": [
                output["facts"] for output in frame_outputs
            ],
            "detr_people_counts": [
                output["detr_people_count"]
                for output in frame_outputs
            ],
            "mean_visual_tag_scores": mean_scores,
        },
    }


In [24]:
def process_verifier_visual(
    records: dict[str, dict[str, Any]],
) -> dict[str, dict[str, Any]]:
    checkpoint_every = 10
    processed_since_checkpoint = 0

    for row in tqdm(
        MANIFEST.itertuples(index=False),
        total=len(MANIFEST),
        desc="Visual verifier",
    ):
        primary = RECORDS.get(row.clip_id)
        if not primary or primary.get("status") != "ok":
            continue
        existing = records.get(row.clip_id, {
            "clip_id": row.clip_id,
            "source_id": row.source_id,
            "source_video": row.source_video,
        })
        if (
            existing.get("visual_status") == "ok"
            and not CFG.force_reprocess
        ):
            continue

        started = time.perf_counter()
        try:
            local_clip = copy_clip_to_local(Path(row.clip_path))
            frames = extract_keyframes(
                local_clip,
                row.clip_id,
                CFG.frame_fractions,
            )
            selection = FRAME_SELECTION_CACHE[row.clip_id]
            frame_outputs = [
                infer_verifier_frame(image)
                for image in frames
            ]
            existing.update({
                "visual_status": "ok",
                "visual": summarize_verifier_frames(
                    frame_outputs,
                    selection["frame_times_sec"],
                    float(selection["duration_sec"]),
                ),
                "frame_sampling": selection,
                "visual_processing_seconds": round(
                    time.perf_counter() - started,
                    3,
                ),
            })
        except Exception as exc:
            existing.update({
                "visual_status": "error",
                "visual_error": {
                    "type": type(exc).__name__,
                    "message": str(exc),
                    "traceback": traceback.format_exc(limit=8),
                },
            })
            if CFG.fail_fast:
                records[row.clip_id] = existing
                save_verifier_records(records)
                raise

        records[row.clip_id] = existing
        processed_since_checkpoint += 1
        if processed_since_checkpoint >= checkpoint_every:
            save_verifier_records(records)
            processed_since_checkpoint = 0

    save_verifier_records(records)

    print(
        "Verifier visual successes:",
        sum(
            record.get("visual_status") == "ok"
            for record in records.values()
        ),
    )
    return records


VERIFIER_RECORDS = process_verifier_visual(VERIFIER_RECORDS)

Visual verifier:   0%|          | 0/626 [00:00<?, ?it/s]

Verifier visual successes: 626


## 13A. Optional Gemini multimodal annotator

Gemini receives the same chronological scene-aware frames used by the local pipeline and returns schema-constrained JSON. It contributes one model-family vote to visual tags, description, setting, activity, indoor/outdoor and people count. Its on-screen text is retained only as a diagnostic comparison: Tesseract and EasyOCR still control the published OCR field.

The request prompt forbids identity inference, hidden-context inference and completion of unreadable text. This reduces hallucination risk but does not eliminate it. API output remains model-generated evidence, not factual ground truth. Requests are sequential to respect rate limits, retried with bounded exponential backoff and checkpointed after each clip.


In [25]:
import base64
from typing import Literal

from google import genai
from pydantic import BaseModel, Field


GEMINI_SOURCE_NAME = f"Gemini {CFG.gemini_model}"
GEMINI_PROMPT_VERSION = "nvtv-gemini-multiframe-v1"
GEMINI_SDK_VERSION = importlib_metadata.version("google-genai")


class GeminiVisualMetadata(BaseModel):
    on_screen_text: list[str] = Field(
        description=(
            "Exact text visibly legible in at least two supplied frames. "
            "Return an empty list when exact characters are uncertain."
        )
    )
    visual_tags: list[str] = Field(
        description="Zero to five exact labels from the supplied vocabulary."
    )
    people_count_numeric: int = Field(
        description=(
            "Typical number of clearly visible people across the frames; "
            "use -1 when a reliable count is impossible."
        )
    )
    uncertainty_notes: list[str] = Field(
        description="Short notes naming fields that could not be read reliably."
    )


GEMINI_PROMPT = f"""
You are an independent visual metadata annotator for an academic video-archive
study. The supplied images are chronological frames from one video clip.

Use visible evidence only. Do not use outside knowledge. Do not identify named
people. Do not infer audio, speech, intent, location names or events that are
not visibly supported. Do not complete cropped, blurred or partly hidden text.

Rules:
1. on_screen_text: copy exact visible characters only when essentially the same
   text is legible in at least two supplied frames. Otherwise return []. Logos
   count only when their letters are clearly readable.
2. visual_tags: choose at most {CFG.tag_top_k} exact strings only from this
   vocabulary: {json.dumps(list(VISUAL_LABELS), ensure_ascii=False)}
3. people_count_numeric: typical clearly visible count across frames; use -1
   when people are obscured, densely crowded or the count varies too much.
4. Put unresolved ambiguity in uncertainty_notes. Do not invent a numeric
   confidence score.
""".strip()

GEMINI_PROMPT_SHA256 = hashlib.sha256(
    GEMINI_PROMPT.encode("utf-8")
).hexdigest()
GEMINI_RESPONSE_SCHEMA = GeminiVisualMetadata.model_json_schema()
GEMINI_RESPONSE_SCHEMA_SHA256 = hashlib.sha256(
    json.dumps(
        GEMINI_RESPONSE_SCHEMA,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()
GEMINI_CONFIG = {
    "schema_version": "1.0",
    "enabled": bool(CFG.use_gemini),
    "provider": "Google Gemini Developer API",
    "model": CFG.gemini_model,
    "sdk_version": GEMINI_SDK_VERSION,
    "prompt_version": GEMINI_PROMPT_VERSION,
    "prompt_sha256": GEMINI_PROMPT_SHA256,
    "response_schema_sha256": GEMINI_RESPONSE_SCHEMA_SHA256,
    "input_policy": "chronological scene-aware frames",
    "frame_sampling": {
        "policy": CFG.frame_sampling_policy,
        "fractions": list(CFG.frame_fractions),
        "max_scene_frames": int(CFG.max_scene_frames),
        "scene_adaptive_threshold": float(CFG.scene_adaptive_threshold),
        "scene_min_length": CFG.scene_min_length,
    },
    "max_frames": int(CFG.gemini_max_frames),
    "image_max_side": int(CFG.gemini_image_max_side),
    "jpeg_quality": int(CFG.gemini_jpeg_quality),
    "temperature": 0.0,
    "thinking_level": "minimal",
    "ocr_role": "diagnostic_only_no_override",
}
GEMINI_SIGNATURE = hashlib.sha256(
    json.dumps(GEMINI_CONFIG, sort_keys=True).encode("utf-8")
).hexdigest()[:12]
GEMINI_FILE = (
    CFG.artifact_dir / f"gemini_predictions_{GEMINI_SIGNATURE}.json"
)


def get_gemini_api_key() -> str:
    """Read the key without printing or persisting it."""
    environment_key = normalize_space(os.environ.get("GEMINI_API_KEY", ""))
    if environment_key:
        return environment_key
    try:
        from google.colab import userdata
        return normalize_space(userdata.get("GEMINI_API_KEY"))
    except Exception:
        return ""


def load_gemini_records() -> dict[str, dict[str, Any]]:
    if not GEMINI_FILE.exists():
        return {}
    try:
        with open(GEMINI_FILE, encoding="utf-8") as handle:
            payload = json.load(handle)
    except (OSError, json.JSONDecodeError) as exc:
        print("Ignoring unreadable Gemini cache:", type(exc).__name__)
        return {}
    if payload.get("gemini_signature") != GEMINI_SIGNATURE:
        return {}
    return {
        record["clip_id"]: record
        for record in payload.get("records", [])
        if isinstance(record, dict) and record.get("clip_id")
    }


def save_gemini_records(records: dict[str, dict[str, Any]]) -> None:
    ordered = [
        records[clip_id]
        for clip_id in MANIFEST["clip_id"]
        if clip_id in records
    ]
    write_json_atomic(
        {
            "schema_version": "1.0",
            "gemini_signature": GEMINI_SIGNATURE,
            "gemini_config": GEMINI_CONFIG,
            "updated_utc": utc_now(),
            "records": ordered,
        },
        GEMINI_FILE,
    )


def select_gemini_frames(
    images: list[Image.Image],
    frame_times: list[float],
) -> tuple[list[Image.Image], list[float], list[int]]:
    if len(images) != len(frame_times):
        raise ValueError("Gemini images and timestamps must align.")
    if not images:
        raise ValueError("Gemini requires at least one frame.")
    maximum = max(1, int(CFG.gemini_max_frames))
    if len(images) <= maximum:
        indices = list(range(len(images)))
    else:
        indices = sorted({
            int(round(index))
            for index in np.linspace(0, len(images) - 1, maximum)
        })
    return (
        [images[index] for index in indices],
        [float(frame_times[index]) for index in indices],
        indices,
    )


def encode_gemini_frame(
    image: Image.Image,
    maximum_side: int,
    jpeg_quality: int,
) -> tuple[dict[str, str], int]:
    prepared = image.convert("RGB").copy()
    if max(prepared.size) > maximum_side:
        prepared.thumbnail(
            (maximum_side, maximum_side),
            Image.Resampling.LANCZOS,
        )
    buffer = io.BytesIO()
    prepared.save(
        buffer,
        format="JPEG",
        quality=int(jpeg_quality),
        optimize=True,
    )
    binary = buffer.getvalue()
    return (
        {
            "type": "image",
            "data": base64.b64encode(binary).decode("ascii"),
            "mime_type": "image/jpeg",
        },
        len(binary),
    )


def build_gemini_input(
    images: list[Image.Image],
    frame_times: list[float],
) -> tuple[list[dict[str, str]], int]:
    maximum_side = int(CFG.gemini_image_max_side)
    jpeg_quality = int(CFG.gemini_jpeg_quality)
    limit_bytes = int(CFG.gemini_inline_request_limit_mb * 1024 * 1024)

    for _ in range(6):
        image_parts, binary_bytes = [], 0
        for image in images:
            part, size = encode_gemini_frame(
                image,
                maximum_side,
                jpeg_quality,
            )
            image_parts.append(part)
            binary_bytes += size
        estimated_request_bytes = int(binary_bytes * 4 / 3) + len(
            GEMINI_PROMPT.encode("utf-8")
        )
        if estimated_request_bytes <= limit_bytes:
            timestamp_text = ", ".join(
                f"frame {index + 1}={time_value:.3f}s"
                for index, time_value in enumerate(frame_times)
            )
            request_parts: list[dict[str, str]] = [{
                "type": "text",
                "text": GEMINI_PROMPT + "\n\nFrame timestamps: " + timestamp_text,
            }]
            for index, part in enumerate(image_parts):
                request_parts.append({
                    "type": "text",
                    "text": f"Chronological frame {index + 1}",
                })
                request_parts.append(part)
            return request_parts, binary_bytes
        maximum_side = max(640, int(maximum_side * 0.80))
        jpeg_quality = max(65, jpeg_quality - 6)

    raise ValueError(
        "Gemini inline request remains too large after compression. "
        "Reduce gemini_max_frames or gemini_image_max_side."
    )


def normalize_gemini_metadata(
    value: GeminiVisualMetadata,
) -> dict[str, Any]:
    raw = value.model_dump(mode="json")
    notes = unique_strings(raw.get("uncertainty_notes", []))

    allowed_tags = {
        normalize_key(label): label for label in VISUAL_LABELS
    }
    selected_tags, discarded_tags = [], []
    for candidate in unique_strings(raw.get("visual_tags", [])):
        canonical = allowed_tags.get(normalize_key(candidate))
        if canonical:
            selected_tags.append(canonical)
        else:
            discarded_tags.append(candidate)
    selected_tags = unique_strings(selected_tags)[:CFG.tag_top_k]
    if discarded_tags:
        notes.append(
            "Discarded out-of-vocabulary tags: "
            + ", ".join(discarded_tags)
        )

    on_screen_text = [
        item for item in unique_strings(raw.get("on_screen_text", []))
        if plausible_ocr_text(item)
    ][:20]

    people = int(raw.get("people_count_numeric", -1))
    if people < 0:
        people_value = None
    elif people > 500:
        people_value = None
        notes.append("Implausible people count was discarded.")
    else:
        people_value = people

    return {
        "on_screen_text": on_screen_text,
        "visual_tags": selected_tags,
        "people_count_numeric": people_value,
        "uncertainty_notes": unique_strings(notes),
    }

def call_gemini_with_retry(
    client: Any,
    images: list[Image.Image],
    frame_times: list[float],
) -> tuple[dict[str, Any], str, Optional[str], int]:
    request_parts, binary_bytes = build_gemini_input(images, frame_times)
    final_error: Optional[Exception] = None
    for attempt in range(max(1, int(CFG.gemini_max_retries))):
        try:
            interaction = client.interactions.create(
                model=CFG.gemini_model,
                input=request_parts,
                response_format={
                    "type": "text",
                    "mime_type": "application/json",
                    "schema": GEMINI_RESPONSE_SCHEMA,
                },
                generation_config={
                    "temperature": 0.0,
                    "thinking_level": "minimal",
                },
            )
            raw_text = str(interaction.output_text or "")
            parsed = GeminiVisualMetadata.model_validate_json(raw_text)
            return (
                normalize_gemini_metadata(parsed),
                raw_text,
                normalize_space(getattr(interaction, "id", "")) or None,
                binary_bytes,
            )
        except Exception as exc:
            final_error = exc
            if attempt + 1 >= max(1, int(CFG.gemini_max_retries)):
                break
            delay = min(
                30.0,
                float(CFG.gemini_retry_base_seconds) * (2 ** attempt),
            )
            print(
                f"Gemini attempt {attempt + 1} failed "
                f"({type(exc).__name__}); retrying in {delay:.1f}s."
            )
            time.sleep(delay)
    assert final_error is not None
    raise final_error


def process_gemini_annotations() -> dict[str, dict[str, Any]]:
    if not CFG.use_gemini:
        print("Gemini integration disabled by configuration.")
        return {}

    records = load_gemini_records()
    api_key = get_gemini_api_key()
    if not api_key:
        print(
            "Gemini API calls skipped: export GEMINI_API_KEY as an environment variable. "
            f"Reusable cached records: {sum(r.get('status') == 'ok' for r in records.values())}."
        )
        return records

    client = genai.Client(api_key=api_key)
    candidates = MANIFEST
    if CFG.gemini_max_clips is not None:
        candidates = candidates.head(int(CFG.gemini_max_clips))

    for row in tqdm(
        candidates.itertuples(index=False),
        total=len(candidates),
        desc="Gemini multimodal annotations",
    ):
        primary = RECORDS.get(row.clip_id)
        if not primary or primary.get("status") != "ok":
            continue
        previous = records.get(row.clip_id)
        if (
            previous
            and previous.get("status") == "ok"
            and previous.get("gemini_signature") == GEMINI_SIGNATURE
            and previous.get("clip_sha256") == row.clip_sha256
            and not CFG.force_reprocess_gemini
        ):
            continue

        started = time.perf_counter()
        try:
            local_clip = copy_clip_to_local(Path(row.clip_path))
            images = extract_keyframes(
                local_clip,
                row.clip_id,
                CFG.frame_fractions,
            )
            selection = FRAME_SELECTION_CACHE[row.clip_id]
            selected_images, selected_times, selected_indices = (
                select_gemini_frames(
                    images,
                    selection["frame_times_sec"],
                )
            )
            metadata, raw_text, interaction_id, payload_bytes = (
                call_gemini_with_retry(
                    client,
                    selected_images,
                    selected_times,
                )
            )
            records[row.clip_id] = {
                "clip_id": row.clip_id,
                "source_id": row.source_id,
                "source_video": row.source_video,
                "clip_sha256": row.clip_sha256,
                "status": "ok",
                "error": None,
                "gemini_signature": GEMINI_SIGNATURE,
                "model": CFG.gemini_model,
                "prompt_version": GEMINI_PROMPT_VERSION,
                "prompt_sha256": GEMINI_PROMPT_SHA256,
                "response_schema_sha256": GEMINI_RESPONSE_SCHEMA_SHA256,
                "processed_utc": utc_now(),
                "processing_seconds": round(
                    time.perf_counter() - started,
                    3,
                ),
                "frame_times_sec": [
                    round(float(value), 4) for value in selected_times
                ],
                "selected_frame_indices": selected_indices,
                "inline_image_payload_bytes": int(payload_bytes),
                "interaction_id": interaction_id,
                "raw_response_sha256": hashlib.sha256(
                    raw_text.encode("utf-8")
                ).hexdigest(),
                "raw_response": raw_text,
                "metadata": metadata,
            }
        except Exception as exc:
            records[row.clip_id] = {
                "clip_id": row.clip_id,
                "source_id": row.source_id,
                "source_video": row.source_video,
                "clip_sha256": row.clip_sha256,
                "status": "error",
                "gemini_signature": GEMINI_SIGNATURE,
                "model": CFG.gemini_model,
                "processed_utc": utc_now(),
                "processing_seconds": round(
                    time.perf_counter() - started,
                    3,
                ),
                "error": {
                    "type": type(exc).__name__,
                    "message": str(exc),
                    "traceback": traceback.format_exc(limit=8),
                },
            }
            save_gemini_records(records)
            if CFG.gemini_fail_fast or CFG.fail_fast:
                raise
        else:
            save_gemini_records(records)
            if CFG.gemini_request_interval_seconds > 0:
                time.sleep(float(CFG.gemini_request_interval_seconds))

    print("Gemini cache:", GEMINI_FILE)
    print(
        "Gemini successes:",
        sum(record.get("status") == "ok" for record in records.values()),
    )
    print(
        "Gemini failures:",
        sum(record.get("status") == "error" for record in records.values()),
    )
    return records


GEMINI_RECORDS = process_gemini_annotations()


Gemini multimodal annotations:   0%|          | 0/626 [00:00<?, ?it/s]

Gemini cache: /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4/gemini_predictions_9fcba08757cd.json
Gemini successes: 626
Gemini failures: 0


## 13B. Perturbation-based robustness experiment

A frozen calibration subset is degraded with blur, reduced brightness and JPEG compression. The verifier is rerun and compared with its original centre-frame evidence. These are stability measurements, not accuracy measurements.

In [26]:
def perturb_image(image: Image.Image, perturbation: str) -> Image.Image:
    image = image.convert("RGB")
    if perturbation == "gaussian_blur":
        return image.filter(ImageFilter.GaussianBlur(radius=2.0))
    if perturbation == "low_brightness":
        return ImageEnhance.Brightness(image).enhance(0.55)
    if perturbation == "jpeg_quality_25":
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=25, optimize=True)
        buffer.seek(0)
        with Image.open(buffer) as compressed:
            return compressed.convert("RGB")
    raise ValueError(f"Unknown perturbation: {perturbation}")


def robustness_set_f1(first: Iterable[str], second: Iterable[str]) -> float:
    first_set = {
        key for value in first
        if (key := normalize_key(value))
    }
    second_set = {
        key for value in second
        if (key := normalize_key(value))
    }
    if not first_set and not second_set:
        return np.nan
    if not first_set or not second_set:
        return 0.0
    overlap = len(first_set & second_set)
    return float(2.0 * overlap / (len(first_set) + len(second_set)))


def robustness_semantic_similarity(first: str, second: str) -> float:
    first, second = normalize_space(first), normalize_space(second)
    if not first or not second:
        return np.nan
    embeddings = VERIFIER_MODELS["semantic_encoder"].encode(
        [first, second],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return float(np.clip(embeddings[0] @ embeddings[1], -1.0, 1.0))


def run_robustness_experiment() -> tuple[pd.DataFrame, pd.DataFrame]:
    candidates = MANIFEST[MANIFEST["split"].eq("calibration")]
    if candidates.empty:
        candidates = MANIFEST
    candidates = candidates.head(CFG.robustness_max_clips)
    rows = []
    perturbations = (
        "gaussian_blur", "low_brightness", "jpeg_quality_25"
    )

    for row in tqdm(
        candidates.itertuples(index=False),
        total=len(candidates),
        desc="Robustness clips",
    ):
        verifier = VERIFIER_RECORDS.get(row.clip_id)
        if not verifier or verifier.get("visual_status") != "ok":
            continue
        local_clip = copy_clip_to_local(Path(row.clip_path))
        frames = extract_keyframes(
            local_clip, row.clip_id, CFG.frame_fractions
        )
        selection = FRAME_SELECTION_CACHE[row.clip_id]
        times = selection["frame_times_sec"]
        duration = float(selection["duration_sec"])
        centre_index = min(
            range(len(times)),
            key=lambda index: abs(times[index] - duration * 0.5),
        )
        centre_image = frames[centre_index]
        original = verifier["visual"]["single_frame"]

        for perturbation in perturbations:
            started = time.perf_counter()
            perturbed_output = infer_verifier_frame(
                perturb_image(centre_image, perturbation)
            )
            perturbed_tags = top_tags(
                perturbed_output["visual_tag_scores"], CFG.tag_top_k
            )
            measures = {
                "ocr_stability": robustness_set_f1(
                    original["on_screen_text"],
                    perturbed_output["on_screen_text"],
                ),
                "tag_stability": robustness_set_f1(
                    original["visual_tags"], perturbed_tags
                ),
                "people_count_stability": float(np.exp(-abs(
                    int(original["detr_people_count"])
                    - int(perturbed_output["detr_people_count"])
                ))),
            }
            finite = [
                value for value in measures.values()
                if np.isfinite(value)
            ]
            rows.append({
                "clip_id": row.clip_id,
                "source_id": row.source_id,
                "perturbation": perturbation,
                "processing_seconds": round(
                    time.perf_counter() - started, 3
                ),
                **measures,
                "mean_stability": (
                    float(np.mean(finite)) if finite else np.nan
                ),
            })

    detail = pd.DataFrame(rows)
    if detail.empty:
        summary = pd.DataFrame()
    else:
        metric_columns = [
            "ocr_stability", "tag_stability", "people_count_stability",
            "mean_stability",
        ]
        summary = (
            detail.groupby("perturbation", as_index=False)
            .agg(
                n_clips=("clip_id", "nunique"),
                **{
                    f"mean_{column}": (column, "mean")
                    for column in metric_columns
                },
            )
        )
    write_csv_atomic(
        detail, CFG.artifact_dir / "robustness_detail.csv"
    )
    write_csv_atomic(
        summary, CFG.artifact_dir / "robustness_summary.csv"
    )
    display(summary.round(4))
    return detail, summary


ROBUSTNESS_DETAIL, ROBUSTNESS_SUMMARY = run_robustness_experiment()

Robustness clips:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pyscenedetect:Detecting scenes...
  if end.get_seconds() > start.get_seconds()

  (start.get_seconds() + end.get_seconds()) / 2.0

INFO:pyscenedetect:Detecting scenes...
  if end.get_seconds() > start.get_seconds()

  (start.get_seconds() + end.get_seconds()) / 2.0

INFO:pyscenedetect:Detecting scenes...
  if end.get_seconds() > start.get_seconds()

  (start.get_seconds() + end.get_seconds()) / 2.0

INFO:pyscenedetect:Detecting scenes...
  if end.get_seconds() > start.get_seconds()

  (start.get_seconds() + end.get_seconds()) / 2.0

INFO:pyscenedetect:Detecting scenes...
  if end.get_seconds() > start.get_seconds()

  (start.get_seconds() + end.get_seconds()) / 2.0



,perturbation,n_clips,mean_ocr_stability,mean_tag_stability,mean_people_count_stability,mean_mean_stability
0,gaussian_blur,5,0.0714,0.96,0.5742,0.5760
1,jpeg_quality_25,5,0.1111,0.80,0.6207,0.5421
2,low_brightness,5,0.3000,0.84,0.5742,0.6303


## 14. Evidence status and field-specific agreement

Every field has a status: `observed`, `not_detected`, `not_applicable`, `conflict` or `error`. Agreement is scored only when at least two non-empty evidence sources exist. Scores exclude self-similarity and quantify corroboration rather than correctness.

The optional `benchmark_reliability_weights.json` file can contain externally estimated positive source weights. If it is absent, model families receive equal weight. Repeated frames do not receive extra voting power.

In [30]:
from difflib import SequenceMatcher


ALLOWED_EVIDENCE_STATUS = {
    "observed", "not_detected", "not_applicable", "conflict", "error"
}
SOURCE_WEIGHTS_FILE = (
    CFG.artifact_dir / "benchmark_reliability_weights.json"
)
DEFAULT_SOURCE_WEIGHTS = {
    "transcript": {
        "Whisper-small": 1.0,
        "Whisper-turbo": 1.0,
    },
    "on_screen_text": {"Tesseract": 1.0, "EasyOCR": 1.0},
    "keywords": {"KeyBERT": 1.0, "YAKE": 1.0, "TF-IDF": 1.0},
    "visual_tags": {
        "CLIP ViT-B/32": 1.0,
        "CLIP ViT-L/14": 1.0,
        GEMINI_SOURCE_NAME: 1.0,
    },
    "people_count": {
        "BLIP VQA": 1.0,
        "ViLT VQA": 1.0,
        "DETR": 1.0,
        GEMINI_SOURCE_NAME: 1.0,
    },
}


def load_source_weights() -> tuple[dict[str, dict[str, float]], str]:
    """Load any compatible reliability weights from the optional JSON file.

    Older benchmark files can contain additional result fields such as
    ``setting`` or ``activity``. Those fields are not source weights, so they
    are reported and ignored instead of stopping the notebook. The same rule
    applies to obsolete source names. Recognized weights are still validated.
    """
    weights = json.loads(json.dumps(DEFAULT_SOURCE_WEIGHTS))
    provenance = "equal_model_family_weights"

    if not SOURCE_WEIGHTS_FILE.exists():
        return weights, provenance

    with open(SOURCE_WEIGHTS_FILE, encoding="utf-8") as handle:
        supplied = json.load(handle)

    if not isinstance(supplied, dict):
        raise ValueError("Reliability weights must be a JSON object.")

    supplied_weights = supplied.get("weights", supplied)
    if not isinstance(supplied_weights, dict):
        raise ValueError("The 'weights' field must be a JSON object.")

    metadata_fields = {
        "description",
        "provenance",
        "version",
        "schema_version",
        "created_at",
        "updated_at",
        "notes",
        "setting",
        "settings",
    }
    ignored_fields: list[str] = []
    ignored_sources: list[str] = []
    applied_weight_count = 0

    for field, source_map in supplied_weights.items():
        if field in metadata_fields:
            continue

        # A benchmark/results JSON may contain fields that are not part of
        # this five-field reliability schema (for example "activity").
        if field not in weights:
            ignored_fields.append(str(field))
            continue

        if not isinstance(source_map, dict):
            raise ValueError(
                f"Weights for '{field}' must be a JSON object mapping "
                "source names to positive numbers."
            )

        for source, raw_value in source_map.items():
            if source not in weights[field]:
                ignored_sources.append(f"{field}/{source}")
                continue

            try:
                value = float(raw_value)
            except (TypeError, ValueError) as exc:
                raise ValueError(
                    f"Weight must be numeric: {field}/{source}={raw_value!r}"
                ) from exc

            if not np.isfinite(value) or value <= 0:
                raise ValueError(
                    f"Weight must be positive and finite: "
                    f"{field}/{source}={raw_value!r}"
                )

            weights[field][source] = value
            applied_weight_count += 1

    if ignored_fields:
        print(
            "Ignored non-weight fields in",
            SOURCE_WEIGHTS_FILE.name + ":",
            ", ".join(sorted(set(ignored_fields))),
        )
    if ignored_sources:
        print(
            "Ignored unrecognized legacy weight sources:",
            ", ".join(sorted(set(ignored_sources))),
        )

    if applied_weight_count:
        provenance = "external_benchmark_weights_file"
        print(
            f"Loaded {applied_weight_count} compatible source weights from",
            SOURCE_WEIGHTS_FILE.name,
        )
    else:
        print(
            "No compatible source weights were found in",
            SOURCE_WEIGHTS_FILE.name + "; using equal defaults.",
        )

    return weights, provenance

SOURCE_WEIGHTS, SOURCE_WEIGHT_PROVENANCE = load_source_weights()
if not SOURCE_WEIGHTS_FILE.exists():
    write_json_atomic(DEFAULT_SOURCE_WEIGHTS, SOURCE_WEIGHTS_FILE)
    print(
        "Created equal-weight template:", SOURCE_WEIGHTS_FILE.name,
        "-- replace values only with independently measured benchmark results."
    )


def source_weight(field: str, source: str) -> float:
    return float(SOURCE_WEIGHTS.get(field, {}).get(source, 1.0))


def bounded(value: float) -> float:
    return float(min(1.0, max(0.0, value)))


def agreement_tier(score: Optional[float]) -> str:
    if score is None or not np.isfinite(score):
        return "not_scored"
    if score >= 0.75:
        return "high_agreement"
    if score >= 0.50:
        return "moderate_agreement"
    return "low_agreement"


def metadata_value_present(value: Any) -> bool:
    if value is None:
        return False
    if isinstance(value, (list, tuple, set, dict)):
        return len(value) > 0
    return bool(normalize_space(value))


def make_consensus_field(
    value: Any,
    score: Optional[float],
    method: str,
    candidates: dict[str, Any],
    support_models: list[str],
    extra: Optional[dict[str, Any]] = None,
    status: Optional[str] = None,
    field_name: Optional[str] = None,
) -> dict[str, Any]:
    inferred_status = (
        "observed" if metadata_value_present(value) else "not_detected"
    )
    status = status or inferred_status
    if status not in ALLOWED_EVIDENCE_STATUS:
        raise ValueError(f"Invalid evidence status: {status}")
    if status != "observed":
        score = None
    clean_score = (
        None
        if score is None or not np.isfinite(score)
        else round(bounded(float(score)), 4)
    )
    tier = agreement_tier(clean_score)
    payload = {
        "value": value,
        "status": status,
        "agreement_score": clean_score,
        "agreement_tier": tier,
        "needs_caution": (
            status in {"conflict", "error"}
            or (
                status == "observed"
                and (clean_score is None or clean_score < 0.50)
            )
        ),
        "method": method,
        "support_models": support_models,
        "source_weights": {
            model: (
                source_weight(field_name, model)
                if field_name else source_weight_from_display_name(model)
            )
            for model in support_models
        },
        "candidates": candidates,
    }
    if extra:
        payload.update(extra)
    return payload


DISPLAY_WEIGHT_LOOKUP = {
    model: weight
    for field_weights in SOURCE_WEIGHTS.values()
    for model, weight in field_weights.items()
}


def source_weight_from_display_name(name: str) -> float:
    return float(DISPLAY_WEIGHT_LOOKUP.get(name, 1.0))


def sample_set_f1(first: set[str], second: set[str]) -> float:
    if not first and not second:
        return np.nan
    if not first or not second:
        return 0.0
    overlap = len(first & second)
    precision = overlap / len(second)
    recall = overlap / len(first)
    return (
        2.0 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )


def pairwise_set_agreement(
    candidate_lists: list[list[str]],
) -> Optional[float]:
    sets = [
        {normalize_key(value) for value in values if normalize_key(value)}
        for values in candidate_lists
    ]
    scores = []
    for first in range(len(sets)):
        for second in range(first + 1, len(sets)):
            if not sets[first] or not sets[second]:
                continue
            scores.append(sample_set_f1(sets[first], sets[second]))
    return float(np.mean(scores)) if scores else None


def ranked_set_consensus(
    candidate_lists: list[list[str]],
    top_k: int,
    weights: Optional[list[float]] = None,
    minimum_source_support: int = 1,
) -> tuple[list[str], Optional[float]]:
    cleaned = [unique_strings(values) for values in candidate_lists]
    weights = weights or [1.0] * len(cleaned)
    if len(weights) != len(cleaned):
        raise ValueError("Set candidate weights do not align.")
    if minimum_source_support < 1:
        raise ValueError("minimum_source_support must be at least one.")
    votes: Counter[str] = Counter()
    rank_scores: Counter[str] = Counter()
    source_support: Counter[str] = Counter()
    canonical: dict[str, str] = {}
    for values, source_score in zip(cleaned, weights):
        for rank, value in enumerate(values):
            key = normalize_key(value)
            if not key:
                continue
            canonical.setdefault(key, value)
            votes[key] += float(source_score)
            rank_scores[key] += float(source_score) / (rank + 1)
            source_support[key] += 1
    ranked = sorted(
        [
            key for key in votes
            if source_support[key] >= minimum_source_support
        ],
        key=lambda key: (-votes[key], -rank_scores[key], key),
    )
    if not ranked:
        return [], None
    return (
        [canonical[key] for key in ranked[:top_k]],
        pairwise_set_agreement(cleaned),
    )


def ocr_canonical_choice(first: str, second: str) -> str:
    candidates = [normalize_space(first), normalize_space(second)]
    def quality(text: str) -> tuple[float, int, int, str]:
        non_space = [character for character in text if not character.isspace()]
        ratio = (
            sum(character.isalnum() for character in non_space)
            / len(non_space)
            if non_space else 0.0
        )
        return ratio, len(normalize_key(text)), len(text.split()), text
    return max(candidates, key=quality)


def strict_ocr_cross_engine_consensus(
    tesseract_lines: list[str],
    easyocr_lines: list[str],
    similarity_threshold: Optional[float] = None,
) -> tuple[list[str], Optional[float], str, dict[str, Any]]:
    threshold = (
        float(getattr(CFG, "ocr_cross_engine_similarity", 0.80))
        if similarity_threshold is None
        else float(similarity_threshold)
    )
    first = unique_strings(tesseract_lines)
    second = unique_strings(easyocr_lines)
    diagnostics = {
        "similarity_threshold": threshold,
        "matched_pairs": [],
        "unmatched_tesseract": [],
        "unmatched_easyocr": [],
    }
    if not first and not second:
        return [], None, "not_detected", diagnostics
    if not first or not second:
        diagnostics["unmatched_tesseract"] = first
        diagnostics["unmatched_easyocr"] = second
        return [], None, "conflict", diagnostics

    pair_scores = sorted([
        (
            SequenceMatcher(
                None, normalize_key(left), normalize_key(right)
            ).ratio(),
            left_index,
            right_index,
        )
        for left_index, left in enumerate(first)
        for right_index, right in enumerate(second)
    ], reverse=True)
    used_first, used_second, selected, similarities = set(), set(), [], []
    for similarity, left_index, right_index in pair_scores:
        if similarity < threshold:
            break
        if left_index in used_first or right_index in used_second:
            continue
        used_first.add(left_index)
        used_second.add(right_index)
        canonical = ocr_canonical_choice(
            first[left_index], second[right_index]
        )
        selected.append(canonical)
        similarities.append(float(similarity))
        diagnostics["matched_pairs"].append({
            "tesseract": first[left_index],
            "easyocr": second[right_index],
            "similarity": round(float(similarity), 4),
            "selected": canonical,
        })

    diagnostics["unmatched_tesseract"] = [
        value for index, value in enumerate(first)
        if index not in used_first
    ]
    diagnostics["unmatched_easyocr"] = [
        value for index, value in enumerate(second)
        if index not in used_second
    ]
    if not selected:
        return [], None, "conflict", diagnostics
    matched_coverage = (
        2.0 * len(selected) / (len(first) + len(second))
    )
    agreement = bounded(float(np.mean(similarities)) * matched_coverage)
    return unique_strings(selected), agreement, "observed", diagnostics




def fuzzy_list_corroboration(
    reference: Iterable[str],
    candidate: Iterable[str],
    threshold: float = 0.80,
) -> dict[str, Any]:
    first = unique_strings(reference)
    second = unique_strings(candidate)
    diagnostics = {
        "threshold": float(threshold),
        "matched_pairs": [],
        "unmatched_reference": [],
        "unmatched_candidate": [],
        "agreement_score": None,
    }
    if not first or not second:
        diagnostics["unmatched_reference"] = first
        diagnostics["unmatched_candidate"] = second
        return diagnostics
    pairs = sorted([
        (
            SequenceMatcher(
                None,
                normalize_key(left),
                normalize_key(right),
            ).ratio(),
            left_index,
            right_index,
        )
        for left_index, left in enumerate(first)
        for right_index, right in enumerate(second)
    ], reverse=True)
    used_first, used_second, scores = set(), set(), []
    for similarity, left_index, right_index in pairs:
        if similarity < threshold:
            break
        if left_index in used_first or right_index in used_second:
            continue
        used_first.add(left_index)
        used_second.add(right_index)
        scores.append(float(similarity))
        diagnostics["matched_pairs"].append({
            "reference": first[left_index],
            "candidate": second[right_index],
            "similarity": round(float(similarity), 4),
        })
    diagnostics["unmatched_reference"] = [
        value for index, value in enumerate(first)
        if index not in used_first
    ]
    diagnostics["unmatched_candidate"] = [
        value for index, value in enumerate(second)
        if index not in used_second
    ]
    if scores:
        coverage = 2.0 * len(scores) / (len(first) + len(second))
        diagnostics["agreement_score"] = round(
            bounded(float(np.mean(scores)) * coverage),
            4,
        )
    return diagnostics


def gemini_candidate_agreement(
    field: str,
    local_value: Any,
    gemini_value: Any,
) -> Optional[float]:
    if field == "on_screen_text":
        return fuzzy_list_corroboration(
            local_value or [],
            gemini_value or [],
            CFG.ocr_cross_engine_similarity,
        )["agreement_score"]
    if field == "visual_tags":
        first = {
            normalize_key(value) for value in (local_value or [])
            if normalize_key(value)
        }
        second = {
            normalize_key(value) for value in (gemini_value or [])
            if normalize_key(value)
        }
        score = sample_set_f1(first, second)
        return None if not np.isfinite(score) else float(score)
    if field == "people_count":
        if local_value is None or gemini_value is None:
            return None
        return float(np.exp(-abs(int(local_value) - int(gemini_value))))
    return None


def values_equal_for_field(field: str, first: Any, second: Any) -> bool:
    if field in {"on_screen_text", "visual_tags"}:
        first_set = {
            normalize_key(value) for value in (first or [])
            if normalize_key(value)
        }
        second_set = {
            normalize_key(value) for value in (second or [])
            if normalize_key(value)
        }
        return first_set == second_set
    if field == "people_count":
        return first == second
    return normalize_key(first) == normalize_key(second)


def gemini_ablation_payload(
    field: str,
    local_value: Any,
    augmented_value: Any,
    gemini_value: Any,
    available: bool,
    policy: str = "one_additional_model_family_vote",
) -> dict[str, Any]:
    if not available:
        return {
            "available": False,
            "policy": policy,
            "local_only_value": local_value,
            "with_gemini_value": augmented_value,
            "gemini_candidate": None,
            "exact_output_changed": False,
            "candidate_vs_local_agreement": None,
            "local_vs_augmented_stability": None,
        }
    return {
        "available": True,
        "policy": policy,
        "local_only_value": local_value,
        "with_gemini_value": augmented_value,
        "gemini_candidate": gemini_value,
        "exact_output_changed": not values_equal_for_field(
            field,
            local_value,
            augmented_value,
        ),
        "candidate_vs_local_agreement": gemini_candidate_agreement(
            field,
            local_value,
            gemini_value,
        ),
        "local_vs_augmented_stability": gemini_candidate_agreement(
            field,
            local_value,
            augmented_value,
        ),
    }


def weighted_median(values: list[int], weights: list[float]) -> int:
    order = np.argsort(values)
    ordered_values = np.asarray(values)[order]
    ordered_weights = np.asarray(weights, dtype=float)[order]
    cutoff = ordered_weights.sum() / 2.0
    index = int(np.searchsorted(np.cumsum(ordered_weights), cutoff))
    return int(ordered_values[min(index, len(ordered_values) - 1)])


def numeric_consensus(
    values: list[Optional[int]],
    weights: Optional[list[float]] = None,
) -> tuple[Optional[int], Optional[float]]:
    original_weights = weights or [1.0] * len(values)
    if len(original_weights) != len(values):
        raise ValueError("Numeric weights do not align.")
    pairs = [
        (int(value), float(weight))
        for value, weight in zip(values, original_weights)
        if value is not None
    ]
    if not pairs:
        return None, None
    if len(pairs) == 1:
        return pairs[0][0], None
    totals: Counter[int] = Counter()
    for value, weight in pairs:
        totals[value] += weight
    ranked = totals.most_common()
    if len(ranked) == 1 or ranked[0][1] > ranked[1][1]:
        selected = int(ranked[0][0])
    else:
        selected = weighted_median(
            [value for value, _ in pairs],
            [weight for _, weight in pairs],
        )
    support = sum(
        weight for value, weight in pairs if value == selected
    )
    return selected, float(support / sum(weight for _, weight in pairs))


def transcript_agreement(
    first: str,
    second: str,
) -> tuple[str, Optional[float], Optional[float]]:
    first = normalize_space(first)
    second = normalize_space(second)
    if not first and not second:
        return "", None, None
    if not first or not second:
        return second or first, None, None
    word_error_rate = float(jiwer.wer(first, second))
    agreement = 1.0 - min(word_error_rate, 1.0)
    return second, agreement, word_error_rate




Ignored non-weight fields in benchmark_reliability_weights.json: activity, indoor_outdoor, language
Loaded 12 compatible source weights from benchmark_reliability_weights.json


In [31]:
FOCUSED_METADATA_FIELDS = (
    "transcript",
    "on_screen_text",
    "keywords",
    "visual_tags",
    "people_count",
)
REQUIRED_DESCRIPTIVE_FIELDS = FOCUSED_METADATA_FIELDS


def build_automatic_ground_truth() -> dict[str, Any]:
    clips = []

    for row in MANIFEST.itertuples(index=False):
        primary = RECORDS.get(row.clip_id)
        verifier = VERIFIER_RECORDS.get(row.clip_id)
        if (
            not primary
            or primary.get("status") != "ok"
            or not verifier
            or verifier.get("text_status") != "ok"
            or verifier.get("visual_status") != "ok"
        ):
            clips.append({
                "clip_id": row.clip_id,
                "source_id": row.source_id,
                "source_video": row.source_video,
                "clip_index": int(row.clip_index),
                "split": row.split,
                "start_sec": float(row.start_sec),
                "end_sec": float(row.end_sec),
                "status": "error",
                "error": "Primary or verifier evidence is incomplete.",
            })
            continue

        common_a = primary["predictions"]["common"]
        single_a = primary["predictions"]["single_frame"]
        multi_a = primary["predictions"]["multi_frame"]
        visual_b = verifier["visual"]
        single_b = visual_b["single_frame"]
        multi_b = visual_b["multi_frame"]
        facts_b = multi_b["facts_per_frame"]

        gemini_record = GEMINI_RECORDS.get(row.clip_id, {})
        gemini_metadata = (
            gemini_record.get("metadata")
            if gemini_record.get("status") == "ok"
            and isinstance(gemini_record.get("metadata"), dict)
            else None
        )
        gemini_available = gemini_metadata is not None

        has_audio = bool(primary["technical"].get("has_audio"))
        if has_audio:
            transcript_value, transcript_score, transcript_wer = (
                transcript_agreement(
                    common_a["transcript"],
                    verifier["transcript_whisper_turbo"],
                )
            )
            transcript_status = None
        else:
            transcript_value, transcript_score, transcript_wer = "", None, None
            transcript_status = "not_applicable"

        ocr_candidates = {
            "Tesseract": multi_a.get("on_screen_text", []),
            "EasyOCR": multi_b.get("on_screen_text", []),
        }
        ocr_names = list(ocr_candidates)
        (
            ocr_value,
            ocr_score,
            ocr_status,
            ocr_matching,
        ) = strict_ocr_cross_engine_consensus(
            ocr_candidates["Tesseract"],
            ocr_candidates["EasyOCR"],
            CFG.ocr_cross_engine_similarity,
        )

        keyword_candidates = {
            "KeyBERT": common_a["keywords"],
            "YAKE": verifier.get("keywords_yake", []),
            "TF-IDF": verifier.get("keywords_tfidf", []),
        }
        keyword_names = list(keyword_candidates)
        keyword_value, keyword_score = ranked_set_consensus(
            list(keyword_candidates.values()),
            top_k=8,
            weights=[
                source_weight("keywords", name)
                for name in keyword_names
            ],
        )

        local_tag_candidates = {
            "CLIP ViT-B/32": multi_a["visual_tags"],
            "CLIP ViT-L/14": multi_b["visual_tags"],
        }
        local_tag_names = list(local_tag_candidates)
        local_tag_value, _ = ranked_set_consensus(
            list(local_tag_candidates.values()),
            top_k=CFG.tag_top_k,
            weights=[
                source_weight("visual_tags", name)
                for name in local_tag_names
            ],
            minimum_source_support=2,
        )
        tag_candidates = dict(local_tag_candidates)
        if gemini_available:
            tag_candidates[GEMINI_SOURCE_NAME] = gemini_metadata.get(
                "visual_tags", []
            )
        tag_names = list(tag_candidates)
        tag_value, tag_score = ranked_set_consensus(
            list(tag_candidates.values()),
            top_k=CFG.tag_top_k,
            weights=[
                source_weight("visual_tags", name)
                for name in tag_names
            ],
            minimum_source_support=2,
        )

        local_people_candidates = {
            "BLIP VQA": [
                single_a.get("people_count_numeric"),
                multi_a.get("people_count_numeric"),
            ],
            "ViLT VQA": [
                parse_people_count(facts["people_count"])
                for facts in facts_b
            ],
            "DETR": multi_b["detr_people_counts"],
        }
        local_people_model_votes = {
            name: numeric_consensus(values)[0]
            for name, values in local_people_candidates.items()
        }
        local_people_names = list(local_people_model_votes)
        local_people_value, _ = numeric_consensus(
            list(local_people_model_votes.values()),
            [
                source_weight("people_count", name)
                for name in local_people_names
            ],
        )
        people_candidates = dict(local_people_candidates)
        if gemini_available:
            people_candidates[GEMINI_SOURCE_NAME] = [
                gemini_metadata.get("people_count_numeric")
            ]
        people_model_votes = {
            name: numeric_consensus(values)[0]
            for name, values in people_candidates.items()
        }
        people_names = list(people_model_votes)
        people_value, people_score = numeric_consensus(
            list(people_model_votes.values()),
            [
                source_weight("people_count", name)
                for name in people_names
            ],
        )

        gemini_ocr_value = (
            gemini_metadata.get("on_screen_text", [])
            if gemini_available else []
        )
        gemini_ocr_diagnostic = fuzzy_list_corroboration(
            ocr_value,
            gemini_ocr_value,
            CFG.ocr_cross_engine_similarity,
        )

        metadata = {
            "transcript": make_consensus_field(
                transcript_value,
                transcript_score,
                "Whisper-small versus Whisper-turbo; turbo is the deterministic tie-break",
                {
                    "Whisper-small": common_a["transcript"],
                    "Whisper-turbo": verifier["transcript_whisper_turbo"],
                },
                ["Whisper-small", "Whisper-turbo"],
                extra={"inter_model_wer": transcript_wer},
                status=transcript_status,
                field_name="transcript",
            ),
            "on_screen_text": make_consensus_field(
                ocr_value,
                ocr_score,
                "strict temporal persistence plus cross-engine fuzzy corroboration",
                {
                    "temporally_persistent": ocr_candidates,
                    "temporal_evidence": {
                        "Tesseract": multi_a.get(
                            "ocr_temporal_evidence", []
                        ),
                        "EasyOCR": multi_b.get(
                            "ocr_temporal_evidence", []
                        ),
                    },
                    "cross_engine_matching": ocr_matching,
                    "centre_frame_diagnostic": {
                        "Tesseract": single_a.get("on_screen_text", []),
                        "EasyOCR": single_b.get("on_screen_text", []),
                    },
                },
                ocr_names,
                extra={
                    "gemini_diagnostic": gemini_ocr_diagnostic,
                    "gemini_ablation": gemini_ablation_payload(
                        "on_screen_text",
                        ocr_value,
                        ocr_value,
                        gemini_ocr_value,
                        gemini_available,
                        policy="diagnostic_only_no_output_override",
                    ),
                },
                status=ocr_status,
                field_name="on_screen_text",
            ),
            "keywords": make_consensus_field(
                keyword_value,
                keyword_score,
                "weighted ranked-set consensus",
                keyword_candidates,
                keyword_names,
                field_name="keywords",
            ),
            "visual_tags": make_consensus_field(
                tag_value,
                tag_score,
                "weighted ranked-set consensus across CLIP architectures",
                {
                    "scene_aware": tag_candidates,
                    "centre_frame_diagnostic": {
                        "CLIP ViT-B/32": single_a["visual_tags"],
                        "CLIP ViT-L/14": single_b["visual_tags"],
                    },
                },
                tag_names,
                extra={
                    "gemini_ablation": gemini_ablation_payload(
                        "visual_tags",
                        local_tag_value,
                        tag_value,
                        gemini_metadata.get("visual_tags", [])
                        if gemini_available else [],
                        gemini_available,
                    )
                },
                field_name="visual_tags",
            ),
            "people_count": make_consensus_field(
                people_value,
                people_score,
                "weighted numeric mode with median tie-break after within-family aggregation",
                {
                    "raw": people_candidates,
                    "model_level_votes": people_model_votes,
                },
                people_names,
                extra={
                    "gemini_ablation": gemini_ablation_payload(
                        "people_count",
                        local_people_value,
                        people_value,
                        gemini_metadata.get("people_count_numeric")
                        if gemini_available else None,
                        gemini_available,
                    )
                },
                field_name="people_count",
            ),
        }

        clips.append({
            "clip_id": row.clip_id,
            "source_id": row.source_id,
            "source_video": row.source_video,
            "clip_index": int(row.clip_index),
            "split": row.split,
            "start_sec": float(row.start_sec),
            "end_sec": float(row.end_sec),
            "status": "ok",
            "ground_truth_metadata": metadata,
        })

    payload = {
        "schema_version": "5.0-focused",
        "dataset": "NVTV 30-second video clips",
        "created_utc": utc_now(),
        "metadata_fields": list(FOCUSED_METADATA_FIELDS),
        "n_clips": len(clips),
        "clips": clips,
    }
    write_json_atomic(payload, GROUND_TRUTH_FILE)

    flat_rows = []
    for clip in clips:
        flat_row = {
            "clip_id": clip["clip_id"],
            "source_id": clip.get("source_id", ""),
            "source_video": clip.get("source_video", ""),
            "clip_index": clip.get("clip_index"),
            "split": clip.get("split", ""),
            "start_sec": clip.get("start_sec", 0.0),
            "end_sec": clip.get("end_sec", 0.0),
            "status": clip.get("status", "error"),
        }
        metadata = clip.get("ground_truth_metadata", {})
        for field in FOCUSED_METADATA_FIELDS:
            evidence = metadata.get(field, {})
            value = (
                evidence.get("value")
                if isinstance(evidence, dict) else evidence
            )
            if isinstance(value, list):
                value = "; ".join(str(item) for item in value)
            flat_row[field] = value
        flat_rows.append(flat_row)
    write_csv_atomic(pd.DataFrame(flat_rows), GROUND_TRUTH_CSV)
    return payload


GROUND_TRUTH = build_automatic_ground_truth()
print("Focused automatic ground truth JSON:", GROUND_TRUTH_FILE)
print("Focused automatic ground truth CSV:", GROUND_TRUTH_CSV)
print("Metadata fields:", ", ".join(FOCUSED_METADATA_FIELDS))
print(
    "Successful clips:",
    sum(clip.get("status") == "ok" for clip in GROUND_TRUTH["clips"]),
)


Focused automatic ground truth JSON: /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4/ground_truth_metadata.json
Focused automatic ground truth CSV: /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4/ground_truth_metadata.csv
Metadata fields: transcript, on_screen_text, keywords, visual_tags, people_count
Successful clips: 626


## 14A. Gemini contribution ablation

This report separates two questions: whether Gemini agrees with the local-only consensus, and whether adding Gemini changes the selected output. Neither quantity is accuracy. OCR is diagnostic-only, so Gemini can disagree with OCR but cannot change its published value.


In [32]:
def build_gemini_ablation_report(
    ground_truth: dict[str, Any],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    for clip in ground_truth.get("clips", []):
        if clip.get("status") != "ok":
            continue
        for field, evidence in clip["ground_truth_metadata"].items():
            diagnostic = evidence.get("gemini_ablation")
            if not isinstance(diagnostic, dict):
                continue
            candidate = diagnostic.get("gemini_candidate")
            rows.append({
                "clip_id": clip["clip_id"],
                "source_id": clip.get("source_id"),
                "split": clip.get("split"),
                "field": field,
                "gemini_available": bool(diagnostic.get("available")),
                "gemini_candidate_present": metadata_value_present(candidate),
                "exact_output_changed": bool(
                    diagnostic.get("exact_output_changed", False)
                ),
                "candidate_vs_local_agreement": diagnostic.get(
                    "candidate_vs_local_agreement"
                ),
                "local_vs_augmented_stability": diagnostic.get(
                    "local_vs_augmented_stability"
                ),
                "policy": diagnostic.get("policy"),
            })

    detail = pd.DataFrame(rows)
    available = (
        detail[detail["gemini_available"]].copy()
        if not detail.empty else pd.DataFrame()
    )
    if available.empty:
        summary = pd.DataFrame(columns=[
            "field", "n_clips", "gemini_candidate_coverage",
            "exact_output_change_rate", "mean_candidate_vs_local_agreement",
            "mean_local_vs_augmented_stability",
        ])
    else:
        summary = (
            available.groupby("field", as_index=False)
            .agg(
                n_clips=("clip_id", "nunique"),
                gemini_candidate_coverage=(
                    "gemini_candidate_present", "mean"
                ),
                exact_output_change_rate=("exact_output_changed", "mean"),
                mean_candidate_vs_local_agreement=(
                    "candidate_vs_local_agreement", "mean"
                ),
                mean_local_vs_augmented_stability=(
                    "local_vs_augmented_stability", "mean"
                ),
            )
            .sort_values("field")
        )

    detail_file = CFG.artifact_dir / "gemini_ablation_detail.csv"
    summary_file = CFG.artifact_dir / "gemini_ablation_summary.csv"
    write_csv_atomic(detail, detail_file)
    write_csv_atomic(summary, summary_file)
    display(summary.round(4))
    if available.empty:
        print(
            "Gemini ablation has no API results. Add GEMINI_API_KEY and rerun "
            "from the Gemini section through the final audit."
        )
    else:
        print("Gemini ablation clips:", available["clip_id"].nunique())
    return detail, summary


GEMINI_ABLATION_DETAIL, GEMINI_ABLATION_SUMMARY = (
    build_gemini_ablation_report(GROUND_TRUTH)
)


,field,n_clips,gemini_candidate_coverage,exact_output_change_rate,mean_candidate_vs_local_agreement,mean_local_vs_augmented_stability
0,on_screen_text,626,0.3850,0.0000,0.5468,1.0000
1,people_count,626,0.6054,0.0176,0.8445,0.9883
2,visual_tags,626,1.0000,0.4904,0.4194,0.9029


Gemini ablation clips: 626


## 15. Focused ground-truth export

This section derives a compact, presentation-facing artifact from the full automatic ground truth. It retains only **transcript**, **on-screen text**, **keywords**, **visual tags**, and **people count** inside each clip’s `ground_truth_metadata` object. Clip identity and timing are retained only so the Flask interface can locate and play each clip. The full research artifact remains unchanged for audit and evaluation.


In [33]:
FOCUSED_GROUND_TRUTH_FIELDS = (
    "transcript",
    "on_screen_text",
    "keywords",
    "visual_tags",
    "people_count",
)
FOCUSED_GROUND_TRUTH_FILE = (
    CFG.artifact_dir / "ground_truth_metadata_focused.json"
)
FOCUSED_GROUND_TRUTH_CSV = (
    CFG.artifact_dir / "ground_truth_metadata_focused.csv"
)


def focused_field_value(item: Any) -> Any:
    """Return a consensus value while tolerating already-flattened input."""
    if isinstance(item, dict) and "value" in item:
        return item.get("value")
    return item


def build_focused_ground_truth(
    payload: dict[str, Any],
) -> dict[str, Any]:
    """Export the five ground-truth fields used by the focused dashboard."""
    focused_clips = []
    flat_rows = []

    for clip in payload.get("clips", []):
        focused_clip = {
            "clip_id": clip.get("clip_id", ""),
            "source_id": clip.get("source_id", ""),
            "source_video": clip.get("source_video", ""),
            "clip_index": clip.get("clip_index"),
            "start_sec": clip.get("start_sec", 0.0),
            "end_sec": clip.get("end_sec", 0.0),
            "status": clip.get("status", "error"),
        }
        if clip.get("error"):
            focused_clip["error"] = clip["error"]

        source_metadata = clip.get("ground_truth_metadata", {})
        if not isinstance(source_metadata, dict):
            source_metadata = {}
        focused_metadata = {
            field: focused_field_value(source_metadata.get(field))
            for field in FOCUSED_GROUND_TRUTH_FIELDS
        }
        focused_clip["ground_truth_metadata"] = focused_metadata
        focused_clips.append(focused_clip)

        row = {
            "clip_id": focused_clip["clip_id"],
            "source_id": focused_clip["source_id"],
            "source_video": focused_clip["source_video"],
            "clip_index": focused_clip["clip_index"],
            "start_sec": focused_clip["start_sec"],
            "end_sec": focused_clip["end_sec"],
            "status": focused_clip["status"],
        }
        for field in FOCUSED_GROUND_TRUTH_FIELDS:
            value = focused_field_value(focused_metadata.get(field))
            if isinstance(value, list):
                value = "; ".join(str(item) for item in value)
            row[field] = value
        flat_rows.append(row)

    focused_payload = {
        "schema_version": "1.0",
        "dataset": payload.get("dataset", "NVTV 30-second video clips"),
        "created_utc": utc_now(),
        "ground_truth_fields": list(FOCUSED_GROUND_TRUTH_FIELDS),
        "n_clips": len(focused_clips),
        "clips": focused_clips,
    }
    write_json_atomic(focused_payload, FOCUSED_GROUND_TRUTH_FILE)
    write_csv_atomic(pd.DataFrame(flat_rows), FOCUSED_GROUND_TRUTH_CSV)
    return focused_payload


FOCUSED_GROUND_TRUTH = build_focused_ground_truth(GROUND_TRUTH)
print("Focused ground truth JSON:", FOCUSED_GROUND_TRUTH_FILE)
print("Focused ground truth CSV:", FOCUSED_GROUND_TRUTH_CSV)
print("Focused fields:", ", ".join(FOCUSED_GROUND_TRUTH_FIELDS))


Focused ground truth JSON: /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4/ground_truth_metadata_focused.json
Focused ground truth CSV: /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4/ground_truth_metadata_focused.csv
Focused fields: transcript, on_screen_text, keywords, visual_tags, people_count


## 16. Read-only Flask focused metadata explorer

Run this section after the focused export above. The interface displays only transcript, on-screen text, keywords, visual tags and people count. Video/clip names and timecodes are retained solely for navigation and playback. Search is restricted to the same five metadata fields.

This interface is deliberately **read-only** and does not modify annotations or claim that automated agreement is factual accuracy. Re-running the cell stops the previous in-notebook server before starting a new one.


In [34]:
from flask import Flask, abort, jsonify, render_template_string, send_from_directory
from werkzeug.serving import make_server
import socket
import threading


DEFAULT_FOCUSED_METADATA_FILE = CFG.artifact_dir / "ground_truth_metadata_focused.json"
DASHBOARD_FIELDS = tuple(globals().get(
    "FOCUSED_GROUND_TRUTH_FIELDS",
    ("transcript", "on_screen_text", "keywords", "visual_tags", "people_count"),
))
DASHBOARD_METADATA_FILE = Path(
    os.environ.get(
        "NVTV_FOCUSED_GROUND_TRUTH_FILE",
        str(globals().get("FOCUSED_GROUND_TRUTH_FILE", DEFAULT_FOCUSED_METADATA_FILE)),
    )
).expanduser()
DASHBOARD_CLIP_DIR = Path(
    os.environ.get("NVTV_CLIP_DIR", str(CFG.clip_dir))
).expanduser()

DASHBOARD_HTML = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>NVTV Focused Ground Truth Metadata</title>
<style>
  :root {
    --bg: #0d1117;
    --panel: #161b22;
    --panel-2: #1f2630;
    --line: #30363d;
    --text: #e6edf3;
    --muted: #8b949e;
    --amber: #f0a93b;
    --amber-soft: rgba(240, 169, 59, .12);
    --teal: #67d4dc;
    --teal-soft: rgba(103, 212, 220, .10);
    --mono: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
    --sans: Inter, ui-sans-serif, system-ui, -apple-system, sans-serif;
  }
  * { box-sizing: border-box; }
  html, body { height: 100%; margin: 0; }
  body {
    background: var(--bg);
    color: var(--text);
    font-family: var(--sans);
    display: flex;
    flex-direction: column;
    overflow: hidden;
  }
  button, input { font: inherit; }
  .topbar {
    display: flex;
    align-items: center;
    gap: 20px;
    padding: 14px 20px;
    border-bottom: 1px solid var(--line);
    background: rgba(13, 17, 23, .96);
  }
  .brand { min-width: 225px; }
  .brand strong {
    color: var(--amber);
    font: 700 18px/1 var(--mono);
    letter-spacing: .15em;
  }
  .brand span {
    display: block;
    margin-top: 5px;
    color: var(--muted);
    font: 10px/1.2 var(--mono);
    letter-spacing: .12em;
  }
  .search-wrap { position: relative; flex: 1; max-width: 520px; }
  #search {
    width: 100%;
    color: var(--text);
    background: var(--panel);
    border: 1px solid var(--line);
    border-radius: 7px;
    padding: 9px 12px;
    outline: none;
  }
  #search:focus { border-color: var(--amber); }
  .search-results {
    position: absolute;
    z-index: 20;
    inset: calc(100% + 6px) 0 auto;
    max-height: 420px;
    overflow: auto;
    padding: 6px;
    background: var(--panel);
    border: 1px solid var(--line);
    border-radius: 8px;
    box-shadow: 0 18px 40px rgba(0, 0, 0, .45);
  }
  .search-results[hidden] { display: none; }
  .result {
    width: 100%;
    padding: 10px;
    color: var(--text);
    text-align: left;
    background: transparent;
    border: 1px solid transparent;
    border-radius: 6px;
    cursor: pointer;
  }
  .result:hover { border-color: var(--amber); background: var(--amber-soft); }
  .result-name { color: var(--amber); font-weight: 650; }
  .result-detail { margin-top: 4px; color: var(--muted); font-size: 12px; }
  .counter { margin-left: auto; color: var(--muted); font: 12px/1 var(--mono); white-space: nowrap; }
  .counter b { color: var(--amber); }
  .shell { min-height: 0; flex: 1; display: flex; }
  .sidebar {
    width: 250px;
    flex: 0 0 250px;
    padding: 12px;
    overflow-y: auto;
    border-right: 1px solid var(--line);
  }
  .video-button {
    width: 100%;
    margin-bottom: 8px;
    padding: 11px;
    color: var(--text);
    text-align: left;
    background: var(--panel);
    border: 1px solid var(--line);
    border-radius: 7px;
    cursor: pointer;
  }
  .video-button:hover, .video-button.active {
    border-color: var(--amber);
    background: var(--amber-soft);
  }
  .video-button small { display: block; margin-top: 5px; color: var(--muted); }
  .main { min-width: 0; min-height: 0; flex: 1; display: flex; flex-direction: column; }
  .stage {
    min-height: 0;
    flex: 1;
    display: grid;
    grid-template-columns: minmax(320px, 1.2fr) minmax(340px, 1fr);
    gap: 18px;
    padding: 18px;
  }
  .viewer { min-width: 0; min-height: 0; display: flex; flex-direction: column; }
  .video-frame { min-height: 0; flex: 1; display: grid; place-items: center; }
  video {
    display: block;
    max-width: 100%;
    max-height: 100%;
    background: #000;
    border: 1px solid var(--line);
    border-radius: 8px;
  }
  .clip-line {
    display: flex;
    justify-content: space-between;
    gap: 12px;
    margin-top: 10px;
    color: var(--muted);
    font: 12px/1.4 var(--mono);
  }
  .clip-line span:first-child { overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
  .metadata {
    min-height: 0;
    overflow-y: auto;
    background: var(--panel);
    border: 1px solid var(--line);
    border-radius: 8px;
  }
  .metadata-title {
    position: sticky;
    top: 0;
    z-index: 2;
    padding: 14px 16px;
    color: var(--amber);
    background: #191d24;
    border-bottom: 1px solid var(--line);
    font: 700 13px/1.35 var(--mono);
  }
  .metadata-body { padding: 2px 16px 18px; }
  .field { padding: 16px 0; border-bottom: 1px solid var(--line); }
  .field:last-child { border-bottom: 0; }
  .label {
    margin-bottom: 9px;
    color: var(--muted);
    font: 10px/1 var(--mono);
    letter-spacing: .15em;
    text-transform: uppercase;
  }
  .transcript {
    padding: 12px 13px;
    color: var(--text);
    background: var(--panel-2);
    border-radius: 6px;
    line-height: 1.6;
    white-space: pre-wrap;
  }
  .tags { display: flex; flex-wrap: wrap; gap: 7px; }
  .tag {
    padding: 5px 9px;
    color: var(--teal);
    background: var(--teal-soft);
    border: 1px solid rgba(103, 212, 220, .35);
    border-radius: 5px;
    font: 11px/1.25 var(--mono);
  }
  .tag.visual { color: var(--amber); background: var(--amber-soft); border-color: rgba(240, 169, 59, .4); }
  .people-count { color: var(--text); font: 700 28px/1 var(--mono); }
  .empty { color: var(--muted); font-style: italic; }
  .strip-wrap { padding: 10px 18px 12px; border-top: 1px solid var(--line); }
  .strip { display: flex; gap: 8px; overflow-x: auto; }
  .clip-button {
    min-width: 78px;
    padding: 8px 10px;
    color: var(--muted);
    background: var(--panel);
    border: 1px solid var(--line);
    border-radius: 6px;
    cursor: pointer;
    font: 11px/1.4 var(--mono);
  }
  .clip-button.active, .clip-button:hover { color: var(--amber); border-color: var(--amber); background: var(--amber-soft); }
  .nav-buttons { display: flex; gap: 7px; }
  .nav-button {
    padding: 7px 10px;
    color: var(--muted);
    background: var(--panel);
    border: 1px solid var(--line);
    border-radius: 6px;
    cursor: pointer;
  }
  .nav-button:disabled { cursor: default; opacity: .35; }
  .empty-page { margin: auto; max-width: 620px; padding: 30px; text-align: center; color: var(--muted); }
  .empty-page h1 { color: var(--text); }
  code { color: var(--amber); font-family: var(--mono); }
  @media (max-width: 900px) {
    body { overflow: auto; }
    .topbar { flex-wrap: wrap; }
    .search-wrap { order: 3; min-width: 100%; }
    .shell { flex-direction: column; }
    .sidebar { width: 100%; flex-basis: auto; display: flex; gap: 8px; border-right: 0; border-bottom: 1px solid var(--line); }
    .video-button { min-width: 190px; margin: 0; }
    .stage { grid-template-columns: 1fr; }
    .video-frame { min-height: 280px; }
    .metadata { max-height: none; }
  }
</style>
</head>
<body>
{% if has_data %}
<header class="topbar">
  <div class="brand">
    <strong>NVTV</strong>
    <span>FOCUSED GROUND TRUTH METADATA</span>
  </div>
  <div class="search-wrap">
    <input id="search" type="search" autocomplete="off"
           placeholder="Search transcript, on-screen text, keywords, visual tags or people count">
    <div id="searchResults" class="search-results" hidden></div>
  </div>
  <div class="counter">
    VIDEO <b id="videoPosition">1/1</b> · CLIP <b id="clipPosition">1/1</b>
  </div>
  <div class="nav-buttons">
    <button id="previous" class="nav-button" aria-label="Previous clip">‹</button>
    <button id="next" class="nav-button" aria-label="Next clip">›</button>
  </div>
</header>

<div class="shell">
  <nav id="videoList" class="sidebar" aria-label="Source videos"></nav>
  <div class="main">
    <main class="stage">
      <section class="viewer">
        <div class="video-frame">
          <video id="player" controls preload="metadata" playsinline></video>
        </div>
        <div class="clip-line">
          <span id="clipFile">—</span>
          <span id="clipTime">—</span>
        </div>
      </section>

      <aside class="metadata">
        <div id="metadataTitle" class="metadata-title">—</div>
        <div class="metadata-body">
          <section class="field">
            <div class="label">Transcript</div>
            <div id="transcript" class="transcript"></div>
          </section>
          <section class="field">
            <div class="label">On-screen text</div>
            <div id="onScreenText" class="tags"></div>
          </section>
          <section class="field">
            <div class="label">Keywords</div>
            <div id="keywords" class="tags"></div>
          </section>
          <section class="field">
            <div class="label">Visual tags</div>
            <div id="visualTags" class="tags"></div>
          </section>
          <section class="field">
            <div class="label">People count</div>
            <div id="peopleCount" class="people-count"></div>
          </section>
        </div>
      </aside>
    </main>
    <footer class="strip-wrap"><div id="clipStrip" class="strip"></div></footer>
  </div>
</div>

<script>
  const CLIPS = {{ clips | tojson }};
  const FIELDS = ["transcript", "on-screen text", "keywords", "visual tags", "people count"];
  const $ = id => document.getElementById(id);
  const groups = [];
  const groupIndex = new Map();
  let videoIndex = 0;
  let clipIndex = 0;

  CLIPS.forEach(clip => {
    const key = clip.source_id || clip.source_video || "Unknown source";
    if (!groupIndex.has(key)) {
      groupIndex.set(key, groups.length);
      groups.push({ key, name: clip.source_video || key, clips: [] });
    }
    groups[groupIndex.get(key)].clips.push(clip);
  });

  function timecode(seconds) {
    const total = Math.max(0, Math.floor(Number(seconds) || 0));
    const h = String(Math.floor(total / 3600)).padStart(2, "0");
    const m = String(Math.floor((total % 3600) / 60)).padStart(2, "0");
    const s = String(total % 60).padStart(2, "0");
    return `${h}:${m}:${s}`;
  }

  function displayItems(element, items, visual = false) {
    element.replaceChildren();
    const values = Array.isArray(items) ? items : (items == null || items === "" ? [] : [items]);
    if (!values.length) {
      const empty = document.createElement("span");
      empty.className = "empty";
      empty.textContent = "None detected";
      element.appendChild(empty);
      return;
    }
    values.forEach(value => {
      const tag = document.createElement("span");
      tag.className = `tag${visual ? " visual" : ""}`;
      tag.textContent = String(value);
      element.appendChild(tag);
    });
  }

  function renderVideoList() {
    $("videoList").replaceChildren();
    groups.forEach((group, index) => {
      const button = document.createElement("button");
      button.className = `video-button${index === videoIndex ? " active" : ""}`;
      button.type = "button";
      button.textContent = group.name;
      const count = document.createElement("small");
      count.textContent = `${group.clips.length} clip${group.clips.length === 1 ? "" : "s"}`;
      button.appendChild(count);
      button.addEventListener("click", () => selectVideo(index));
      $("videoList").appendChild(button);
    });
  }

  function renderClipStrip() {
    $("clipStrip").replaceChildren();
    groups[videoIndex].clips.forEach((clip, index) => {
      const button = document.createElement("button");
      button.type = "button";
      button.className = `clip-button${index === clipIndex ? " active" : ""}`;
      button.textContent = `${String(index + 1).padStart(2, "0")}\n${timecode(clip.start_sec)}`;
      button.addEventListener("click", () => selectClip(index));
      $("clipStrip").appendChild(button);
    });
  }

  function renderClip() {
    const group = groups[videoIndex];
    const clip = group.clips[clipIndex];
    const duration = Math.max(0, Number(clip.end_sec) - Number(clip.start_sec));
    $("player").src = `/clips/${encodeURIComponent(clip.clip)}`;
    $("clipFile").textContent = clip.clip;
    $("clipTime").textContent = `${timecode(clip.start_sec)}–${timecode(clip.end_sec)} · ${Math.round(duration)}s`;
    $("metadataTitle").textContent = clip.clip;

    const transcript = String(clip.transcript || "").trim();
    $("transcript").textContent = transcript || "No speech detected";
    $("transcript").classList.toggle("empty", !transcript);
    displayItems($("onScreenText"), clip.on_screen_text);
    displayItems($("keywords"), clip.keywords);
    displayItems($("visualTags"), clip.visual_tags, true);
    $("peopleCount").textContent = clip.people_count == null || clip.people_count === ""
      ? "Not detected"
      : String(clip.people_count);
    $("peopleCount").classList.toggle("empty", clip.people_count == null || clip.people_count === "");

    $("videoPosition").textContent = `${videoIndex + 1}/${groups.length}`;
    $("clipPosition").textContent = `${clipIndex + 1}/${group.clips.length}`;
    $("previous").disabled = clipIndex === 0;
    $("next").disabled = clipIndex === group.clips.length - 1;
    document.querySelectorAll(".clip-button").forEach((button, index) => {
      button.classList.toggle("active", index === clipIndex);
    });
  }

  function selectVideo(index) {
    videoIndex = Math.max(0, Math.min(groups.length - 1, index));
    clipIndex = 0;
    renderVideoList();
    renderClipStrip();
    renderClip();
  }

  function selectClip(index) {
    clipIndex = Math.max(0, Math.min(groups[videoIndex].clips.length - 1, index));
    renderClip();
  }

  $("previous").addEventListener("click", () => selectClip(clipIndex - 1));
  $("next").addEventListener("click", () => selectClip(clipIndex + 1));

  const searchRows = [];
  groups.forEach((group, vi) => group.clips.forEach((clip, ci) => {
    const values = {
      "transcript": String(clip.transcript || ""),
      "on-screen text": (clip.on_screen_text || []).join(" "),
      "keywords": (clip.keywords || []).join(" "),
      "visual tags": (clip.visual_tags || []).join(" "),
      "people count": clip.people_count == null ? "" : String(clip.people_count),
    };
    searchRows.push({ vi, ci, clip, values, text: Object.values(values).join(" ").toLowerCase() });
  }));

  function showSearchResults() {
    const terms = $("search").value.trim().toLowerCase().split(/\s+/).filter(Boolean);
    const box = $("searchResults");
    box.replaceChildren();
    if (!terms.length) { box.hidden = true; return; }
    const matches = searchRows.filter(row => terms.every(term => row.text.includes(term))).slice(0, 100);
    if (!matches.length) {
      const empty = document.createElement("div");
      empty.className = "result-detail";
      empty.style.padding = "12px";
      empty.textContent = "No matching metadata";
      box.appendChild(empty);
    }
    matches.forEach(row => {
      const button = document.createElement("button");
      button.type = "button";
      button.className = "result";
      const name = document.createElement("div");
      name.className = "result-name";
      name.textContent = row.clip.source_video || row.clip.clip;
      const matchedFields = FIELDS.filter(field => terms.some(term => row.values[field].toLowerCase().includes(term)));
      const detail = document.createElement("div");
      detail.className = "result-detail";
      detail.textContent = `${row.clip.clip} · ${matchedFields.join(", ")}`;
      button.append(name, detail);
      button.addEventListener("click", () => {
        videoIndex = row.vi;
        clipIndex = row.ci;
        renderVideoList();
        renderClipStrip();
        renderClip();
        box.hidden = true;
      });
      box.appendChild(button);
    });
    box.hidden = false;
  }

  $("search").addEventListener("input", showSearchResults);
  document.addEventListener("click", event => {
    if (!event.target.closest(".search-wrap")) $("searchResults").hidden = true;
  });
  document.addEventListener("keydown", event => {
    if (event.target === $("search")) return;
    if (event.key === "ArrowLeft") selectClip(clipIndex - 1);
    if (event.key === "ArrowRight") selectClip(clipIndex + 1);
    if (event.key === "ArrowUp") selectVideo(videoIndex - 1);
    if (event.key === "ArrowDown") selectVideo(videoIndex + 1);
  });

  selectVideo(0);
</script>
{% else %}
<div class="empty-page">
  <h1>No focused ground-truth metadata found</h1>
  <p>Run the focused export cell first. The dashboard is reading <code>{{ metadata_path }}</code>.</p>
</div>
{% endif %}
</body>
</html>'''


def dashboard_load_payload() -> Optional[dict[str, Any]]:
    if not DASHBOARD_METADATA_FILE.exists():
        return None
    with open(DASHBOARD_METADATA_FILE, encoding="utf-8") as handle:
        payload = json.load(handle)
    if not isinstance(payload, dict) or not isinstance(payload.get("clips"), list):
        raise ValueError(
            "The dashboard expects ground_truth_metadata_focused.json."
        )
    return payload


def dashboard_value(
    metadata: dict[str, Any],
    field: str,
    default: Any = "",
) -> Any:
    item = metadata.get(field, default)
    if isinstance(item, dict):
        item = item.get("value", default)
    return default if item is None else item


def dashboard_list(value: Any) -> list[str]:
    if value is None or value == "":
        return []
    if isinstance(value, (list, tuple, set)):
        return [str(item) for item in value if normalize_space(item)]
    return [str(value)]


def dashboard_clip_record(clip: dict[str, Any]) -> dict[str, Any]:
    metadata = clip.get("ground_truth_metadata", {})
    if not isinstance(metadata, dict):
        metadata = {}
    clip_id = str(clip.get("clip_id", ""))
    return {
        "clip": f"{clip_id}.mp4",
        "clip_id": clip_id,
        "source_id": clip.get("source_id", ""),
        "source_video": clip.get("source_video", ""),
        "start_sec": float(clip.get("start_sec", 0.0) or 0.0),
        "end_sec": float(clip.get("end_sec", 0.0) or 0.0),
        "transcript": str(dashboard_value(metadata, "transcript", "")),
        "on_screen_text": dashboard_list(
            dashboard_value(metadata, "on_screen_text", [])
        ),
        "keywords": dashboard_list(
            dashboard_value(metadata, "keywords", [])
        ),
        "visual_tags": dashboard_list(
            dashboard_value(metadata, "visual_tags", [])
        ),
        "people_count": dashboard_value(metadata, "people_count", None),
    }


def dashboard_records(payload: Optional[dict[str, Any]]) -> list[dict[str, Any]]:
    if not payload:
        return []
    return [dashboard_clip_record(clip) for clip in payload.get("clips", [])]


dashboard_app = Flask("nvtv_focused_ground_truth_dashboard")


@dashboard_app.after_request
def dashboard_headers(response):
    response.headers["Cache-Control"] = "no-store"
    response.headers["X-Content-Type-Options"] = "nosniff"
    return response


@dashboard_app.route("/")
def dashboard_index():
    payload = dashboard_load_payload()
    clips = dashboard_records(payload)
    return render_template_string(
        DASHBOARD_HTML,
        clips=clips,
        metadata_path=str(DASHBOARD_METADATA_FILE),
        has_data=bool(clips),
    )


@dashboard_app.route("/api/health")
def dashboard_health():
    payload = dashboard_load_payload()
    clips = dashboard_records(payload)
    return jsonify({
        "status": "ok" if payload else "metadata_missing",
        "schema_version": payload.get("schema_version") if payload else None,
        "clip_count": len(clips),
    })


@dashboard_app.route("/api/ground-truth")
def dashboard_ground_truth():
    payload = dashboard_load_payload()
    if payload is None:
        return jsonify({
            "error": "ground_truth_metadata_focused.json not found",
            "path": str(DASHBOARD_METADATA_FILE),
        }), 404
    return jsonify(payload)


@dashboard_app.route("/api/clips")
def dashboard_clips():
    return jsonify(dashboard_records(dashboard_load_payload()))


@dashboard_app.route("/clips/<path:filename>")
def dashboard_serve_clip(filename: str):
    payload = dashboard_load_payload()
    allowed = {
        f"{clip.get('clip_id', '')}.mp4"
        for clip in (payload or {}).get("clips", [])
    }
    if filename not in allowed or not DASHBOARD_CLIP_DIR.exists():
        abort(404)
    return send_from_directory(
        str(DASHBOARD_CLIP_DIR),
        filename,
        conditional=True,
    )


class DashboardServerThread(threading.Thread):
    def __init__(self, flask_app: Flask):
        super().__init__(daemon=True)
        self.server = make_server(
            "127.0.0.1", 0, flask_app, threaded=True
        )
        self.port = int(self.server.server_port)

    def run(self) -> None:
        self.server.serve_forever()

    def stop(self) -> None:
        self.server.shutdown()


previous_server = globals().get("NVTV_DASHBOARD_SERVER")
if previous_server is not None:
    try:
        previous_server.stop()
    except Exception as exc:
        print("Previous dashboard shutdown warning:", exc)

NVTV_DASHBOARD_SERVER = DashboardServerThread(dashboard_app)
NVTV_DASHBOARD_SERVER.start()
DASHBOARD_PORT = NVTV_DASHBOARD_SERVER.port

for _ in range(40):
    try:
        with socket.create_connection(("127.0.0.1", DASHBOARD_PORT), timeout=0.5):
            break
    except OSError:
        time.sleep(0.25)
else:
    raise RuntimeError("The Flask dashboard did not start within 10 seconds.")

print("Dashboard metadata:", DASHBOARD_METADATA_FILE)
print("Dashboard clips:", DASHBOARD_CLIP_DIR)
print("Displayed fields:", ", ".join(DASHBOARD_FIELDS))
print("Flask:", importlib_metadata.version("flask"))

try:
    from google.colab import output as colab_output
except ImportError:
    print(f"Open http://127.0.0.1:{DASHBOARD_PORT}/")
else:
    colab_output.serve_kernel_port_as_iframe(
        DASHBOARD_PORT,
        path="/",
        height="900",
    )


Dashboard metadata: /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4/ground_truth_metadata_focused.json
Dashboard clips: /content/drive/MyDrive/NVTV_PublicData1/automatic_ground_truth_v4/clips
Displayed fields: transcript, on_screen_text, keywords, visual_tags, people_count
Flask: 3.1.3


<IPython.core.display.Javascript object>